# Control-arm crossover, Western enrollment and overall survival in front-line registrational cancer trials

Cross-sectional analysis of phase 3, registrational, front-line (first-line metastatic or adjuvant) cancer
trials in which the experimental drug already had an FDA-approved later-line indication for the same disease.
For each trial the notebook relates control-arm crossover (the share of control patients who later received
the experimental drug, or a drug of its class) and Western enrollment (the share of participants enrolled in
North America or Europe) to the overall-survival hazard ratio, using inverse-variance-weighted random-effects
meta-regression (REML, Hartung-Knapp), plus descriptive stratified correlations and sensitivity analyses.

**Data source.** `data/Crossover_Trials_Master.xlsx`, the abstracted analysis dataset (one row per comparison;
see `data/Data_Dictionary.csv`). It is built from the HemOnc.org Knowledge Base and the FDA Oncology/Hematologic
Malignancies Approval Notifications; see the repository README.

**Run all cells from a fresh kernel; outputs are written to `outputs/`.** They are the manuscript tables
(`Manuscript_Tables.xlsx` and `Manuscript_Tables.docx`: Tables 1-2 and Supplementary Tables S1-S10) and
Figure 2 (`outputs/figures/Figure_2.png` and `.pdf`).

## 1. Imports
Dependencies are pinned in `requirements.txt`; the notebook installs nothing itself.

In [1]:
import os
import re
import shutil
import zipfile
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import openpyxl
import pandas as pd
from docx import Document
from docx.enum.section import WD_ORIENT, WD_SECTION
from docx.enum.table import WD_ALIGN_VERTICAL
from docx.enum.text import WD_ALIGN_PARAGRAPH, WD_LINE_SPACING
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Inches, Pt
from docx.text.paragraph import Paragraph
from openpyxl.cell.rich_text import CellRichText, TextBlock
from openpyxl.cell.text import InlineFont
from openpyxl.styles import Alignment, Border, Font, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.page import PageMargins
from openpyxl.worksheet.properties import PageSetupProperties
from scipy import stats

# Paths in this notebook are relative to the repository root, wherever the kernel starts.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "Crossover_Trials_Master.xlsx").exists())
os.chdir(ROOT)

# ── Figure font ──────────────────────────────────────────────────────────────
# Fonts are resolved explicitly: the first family below that is installed wins, and the choice is printed
# rather than substituted silently.  A machine without Arial still runs, but its figures will not match
# the submitted ones exactly, and the printed line says so.

import matplotlib
import matplotlib.font_manager as fm

FIGURE_FONT_PREFERENCE = ["Arial", "Helvetica", "DejaVu Sans"]
_installed = {f.name for f in fm.fontManager.ttflist}
FIGURE_FONT = next((f for f in FIGURE_FONT_PREFERENCE if f in _installed), "DejaVu Sans")
if FIGURE_FONT != FIGURE_FONT_PREFERENCE[0]:
    print(f"NOTE: '{FIGURE_FONT_PREFERENCE[0]}' is not installed; figures will render in "
          f"'{FIGURE_FONT}' and will not match the submitted figures exactly.")
else:
    print(f"Figure font: {FIGURE_FONT}")

matplotlib.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": [FIGURE_FONT] + [f for f in FIGURE_FONT_PREFERENCE if f != FIGURE_FONT],
    # keep mathtext (the italic P in Figure 2) in the same family as body text
    "mathtext.fontset": "custom",
    "mathtext.rm": FIGURE_FONT,
    "mathtext.it": f"{FIGURE_FONT}:italic",
    "mathtext.bf": f"{FIGURE_FONT}:bold",
    "pdf.fonttype": 42,          # embed TrueType text in the PDF so it stays editable
    # figure style used for Figure 2
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "xtick.direction": "out",
    "ytick.direction": "out",
    "axes.edgecolor": "#333333",
    "axes.linewidth": 0.8,
})

import matplotlib.pyplot as plt


Figure font: Arial


## 2. Configuration
Input file, the counts this dataset is expected to produce, the table-numbering registry, and the in-memory
workbook that every table section writes into.

In [2]:
MASTER_FILE = "data/Crossover_Trials_Master.xlsx"
SHEET = "Final Study List"
RESULTS_WORKBOOK = "outputs/Manuscript_Tables.xlsx"
DOCX_PATH = "outputs/Manuscript_Tables.docx"
FIGURES_DIR = "outputs/figures"

# Expected counts for the current dataset (update deliberately if the source data change).
# The assertions that read these are tripwires: a change in the data should stop the run.
EXPECTED = {
    "n_over_exact": 2,            # exact-drug progressed-denominator rates > 100%
    "n_over_inclass": 4,          # in-class progressed-denominator rates > 100%
    "n_impossible_rows": 4,       # distinct comparisons in Supplementary Table S1
    "n_excluded_impossible": 4,   # excluded comparisons with a rate > 100%
    "n_excluded_total": 15,       # all excluded-comparison records (internal check)
    "n_stratified": 289,          # computable stratified correlations
    "n_stratified_by_assoc": [66, 32, 191],
    "n_nominal_sig": 7,           # stratified correlations with unadjusted P < .05
}

# Table numbering lives here and nowhere else.  Sheets are addressed by a semantic key; the tab name
# carries the number, and TABLE_LABEL derives the label that captions and footnotes quote
# ("Table 1", "Supplementary Table S4") from that tab name.  Renumbering a table means editing one
# string in this dictionary.
SHEET_NAMES = {
    "T_XOVER":      "T1. Crossover vs OS",
    "T_GEO":        "T2. Geo vs OS",
    "S_IMPOSSIBLE": "S1. Impossible Crossover",
    "S_SUMMARY":    "S2. Summary of Studies",
    "S_STUDIES":    "S3. All Included Studies",
    "S_XOVER_FULL": "S4. Meta-reg Crossover vs OS",
    "S_HETERO":     "S5. Heterogeneity",
    "S_BOUNDS":     "S6. Precision Bounds",
    "S_GEO_FULL":   "S7. Meta-reg Geo vs OS",
    "S_STRAT_ALL":  "S8. All Stratified Correlations",
    "S_COLLAPSED":  "S9. Collapsed Sensitivity",
    "S_CAPPED":     "S10. Crossover Capped at 100%",
}


def _table_label_from_sheet(sheet_name):
    """'T1. Crossover vs OS' -> 'Table 1'; 'S4. Heterogeneity' -> 'Supplementary Table S4'."""
    _m = re.match(r"^([TS])(\d+)\.", sheet_name)
    assert _m, f"Numbered sheet name must start with T<n>. or S<n>.: {sheet_name!r}"
    return f"Table {_m.group(2)}" if _m.group(1) == "T" else f"Supplementary Table S{_m.group(2)}"


TABLE_LABEL = {_key: _table_label_from_sheet(_name) for _key, _name in SHEET_NAMES.items()}

# ── Multi-row trials: which row is the trial ────────────────────────────────
# Eight trials contribute two rows each to the 86-comparison dataset, for two different reasons.  The
# keep-lists live in the configuration because the study-level tables and the collapsed sensitivity
# analysis must designate the same primary agent.
#
# Arm-split trials: two genuinely different experimental arms against a shared control.  The keep-list
# retains the registrational/guideline-preferred arm.
ARM_SPLIT_DROPS = {
    "CheckMate 067": "Nivolumab monotherapy",
    "CheckMate 227": "Nivolumab monotherapy",
    "EMBARK": "Enzalutamide monotherapy",
    "IMvigor130": "Atezolizumab monotherapy",
}
# Target-split trials: one comparison, entered twice because two drugs of the
# experimental regimen each carried a qualifying prior later-line approval.  The
# keep-list retains the component whose approval qualified the trial -- that
# retained row is the trial's primary agent.
TARGET_SPLIT_DROPS = {
    "CLEAR": "Lenvatinib",
    "RENOTORCH": "Axitinib",
    "COMBI-AD": "Trametinib",
    "COMBI-v": "Trametinib",
}

# Crossover columns: numerator (exact drug / drug in class) x denominator (subsequent therapy / progressed / event).
col_exact_sub = "Frac_ctrl_pts_getting_exact_drug_post-protocol_progressed_subsequent_therapy"
col_inclass_sub = "Frac_ctrl_pts_getting_post-protocol_drug_in_class_progressed_subsequent_therapy"
col_exact_prog = "Frac_ctrl_pts_getting_exact_drug_post-protocol_progressed"
col_inclass_prog = "Frac_ctrl_pts_getting_post-protocol_drug_in_class_progressed"
col_exact_event = "Frac_ctrl_pts_getting_exact_drug_post-protocol_event"
col_inclass_event = "Frac_ctrl_pts_getting_post-protocol_drug_in_class_event"

# One workbook, filled section by section and written once, at the end.
wb = openpyxl.Workbook()
wb.remove(wb.active)
os.makedirs(FIGURES_DIR, exist_ok=True)

## 3. Setup and helper functions
Every helper used by more than one section is defined here, once. The geography parser, the start-year
parser, the clinical-setting grouping and the drug-class grouping each had several near-copies in earlier
versions of this analysis; the copies that fed published tables were line-for-line equivalent, so they are
merged. (`group_drug_class_for_regression` had a second variant with short labels that only the discarded
exploratory plots used.)

In [3]:
# ── Geography: a percentage cell -> a number on the 0-100 scale ─────────────────
def parse_geo_pct(val):
    if pd.isna(val):
        return np.nan
    if isinstance(val, (int, float)):
        if val == 0:
            return 0.0
        elif val <= 1.0:
            return val * 100.0
        else:
            return float(val)
    val_str = str(val).strip().lower()
    if val_str in ["na", "no data", "nr", "nan", ""]:
        return np.nan
    match = re.search(r"([0-9.]+)\s*%", val_str)
    if match:
        return float(match.group(1))
    match_num = re.search(r"([0-9.]+)", val_str)
    if match_num:
        num = float(match_num.group(1))
        if num <= 1.0:
            return num * 100.0
        else:
            return num
    return np.nan


# ── Enrollment start year ──────────────────────────────────────────────────────
def get_start_year(row):
    start_val = row.get("start")
    if pd.notna(start_val) and not isinstance(start_val, str):
        return start_val.year
    if pd.notna(start_val) and isinstance(start_val, str) and str(start_val).strip() != "":
        try:
            return pd.to_datetime(start_val).year
        except:
            pass
    enroll_val = row.get("enrollment")
    if pd.notna(enroll_val):
        match = re.search(r"(\d{4})", str(enroll_val))
        if match:
            return int(match.group(1))
    return np.nan


# ── Clinical setting and drug-class groupings used by the stratified analyses ──
def group_context(c):
    c_str = str(c).lower()
    if "adjuvant" in c_str or "neoadjuvant" in c_str:
        return "Adjuvant/Neoadjuvant"
    else:
        return "First-line (Metastatic/Advanced)"


def group_drug_class_for_regression(dc):
    if dc == "Immunotherapy (CPI-based)": return "Immunotherapy (CPI-based)"
    elif dc == "Small-Molecule Targeted Therapy": return "Small-Molecule Targeted Therapy"
    else: return "Other"


# ── Drug class is keyed on the agent under study, not on the full regimen string ──
# Drug class is keyed on the agent under study -- the drug with the qualifying prior later-line approval,
# i.e. the drug the exact-drug crossover numerator counts -- not on the full regimen string, which
# misassigns combination regimens.
SPECIFIC_TREATMENT_TO_CLASS = {
    # Immune checkpoint inhibitors
    "Pembrolizumab": "Immunotherapy (CPI-based)",
    "Nivolumab": "Immunotherapy (CPI-based)",
    "Atezolizumab": "Immunotherapy (CPI-based)",
    "Durvalumab": "Immunotherapy (CPI-based)",
    "Avelumab": "Immunotherapy (CPI-based)",
    "Dostarlimab": "Immunotherapy (CPI-based)",
    "Toripalimab": "Immunotherapy (CPI-based)",
    # Small-molecule targeted therapy
    "Osimertinib": "Small-Molecule Targeted Therapy",
    "Erlotinib": "Small-Molecule Targeted Therapy",
    "Alectinib": "Small-Molecule Targeted Therapy",
    "Ceritinib": "Small-Molecule Targeted Therapy",
    "Crizotinib": "Small-Molecule Targeted Therapy",
    "Dabrafenib": "Small-Molecule Targeted Therapy",
    "Trametinib": "Small-Molecule Targeted Therapy",
    "Encorafenib": "Small-Molecule Targeted Therapy",
    "Sotorasib": "Small-Molecule Targeted Therapy",
    "Ivosidenib": "Small-Molecule Targeted Therapy",
    "Venetoclax": "Small-Molecule Targeted Therapy",
    "Ibrutinib": "Small-Molecule Targeted Therapy",
    "Acalabrutinib": "Small-Molecule Targeted Therapy",
    "Olaparib": "Small-Molecule Targeted Therapy",
    "Dasatinib": "Small-Molecule Targeted Therapy",
    "Everolimus": "Small-Molecule Targeted Therapy",
    "Abemaciclib": "Small-Molecule Targeted Therapy",
    "Lenvatinib": "Small-Molecule Targeted Therapy",
    "Axitinib": "Small-Molecule Targeted Therapy",
    # Monoclonal antibodies (non-CPI)
    "Daratumumab": "Monoclonal Antibody (non-CPI)",
    "Isatuximab": "Monoclonal Antibody (non-CPI)",
    "Pertuzumab": "Monoclonal Antibody (non-CPI)",
    "Ofatumumab": "Monoclonal Antibody (non-CPI)",
    "Bevacizumab": "Monoclonal Antibody (non-CPI)",
    "Ramucirumab": "Monoclonal Antibody (non-CPI)",
    "Amivantamab": "Monoclonal Antibody (non-CPI)",
    # Antibody-drug conjugates
    "Brentuximab Vedotin": "Antibody-Drug Conjugate (ADC)",
    "Trastuzumab emtansine": "Antibody-Drug Conjugate (ADC)",
    # Hormonal / endocrine
    "Enzalutamide": "Hormonal / Endocrine Therapy",
    "Abiraterone": "Hormonal / Endocrine Therapy",
    # Immunomodulatory / proteasome inhibitor  (NEW CLASS)
    "Lenalidomide": "Immunomodulatory / Proteasome Inhibitor",
    "Bortezomib": "Immunomodulatory / Proteasome Inhibitor",
    # Chemotherapy / cytotoxics
    "Cabazitaxel": "Chemotherapy / Cytotoxics",
    "Liposomal Irinotecan": "Chemotherapy / Cytotoxics",
    "Platinum-based chemotherapy": "Chemotherapy / Cytotoxics",
    # Radioligand
    "177Lu-PSMA-617": "Radioligand Therapy",
}


def classify_drug_class_from_specific(specific_treatment):
    """Map the Specific Treatment cell to a drug class.

    Deliberately raises on an unrecognised value rather than falling back to 'Other/Unknown':
    a typo or a newly added trial must fail loudly here.
    """
    key = str(specific_treatment).strip()
    if key not in SPECIFIC_TREATMENT_TO_CLASS:
        raise KeyError(
            f"Specific Treatment {key!r} is not in SPECIFIC_TREATMENT_TO_CLASS. "
            "Add it to the map rather than letting it fall through to Other/Unknown."
        )
    return SPECIFIC_TREATMENT_TO_CLASS[key]


def collapse_to_one_row_per_trial(data_df):
    """One row per trial, keeping the primary agent for target-split trials.

    Four trials contribute two rows that differ only in which drug of the
    experimental regimen carried the qualifying prior later-line approval.
    Study-level tables (S2 and S3) need exactly one drug class per trial, so a trial whose rows straddle
    two drug classes is reduced to the row of its primary agent.

    The primary agent is the row TARGET_SPLIT_DROPS retains, the same designation the collapsed sensitivity
    analysis (S9) uses: the immune checkpoint inhibitor for the two CPI-plus-TKI combinations (CLEAR ->
    pembrolizumab, RENOTORCH -> toripalimab), and dabrafenib for the two dabrafenib/trametinib trials.

    Arm-split trials (CheckMate 067, CheckMate 227, EMBARK, IMvigor130) carry
    the same Specific Treatment on both rows, so keep="first" is safe for them.
    """
    study = data_df["study"].astype(str).str.strip()
    specific = data_df["Specific Treatment"].astype(str).str.strip()
    drop_mask = pd.Series(False, index=data_df.index)
    for trial, dropped_agent in TARGET_SPLIT_DROPS.items():
        m = (study == trial) & (specific == dropped_agent)
        assert m.sum() == 1, (
            f"Expected exactly 1 non-primary row for {trial} ({dropped_agent}), found {m.sum()}"
        )
        drop_mask |= m
    return data_df[~drop_mask].drop_duplicates(subset=["study"], keep="first").copy()


PCT_SCALE_FOOTNOTE = (
    "Effects are percentage changes in the OS hazard ratio over the stated increment of the "
    "exposure: 100 × (ratio − 1) for the exponentiated meta-regression coefficient. A value "
    "of +0.18% corresponds to a hazard ratio of 1.0018."
)
# Tables 1 and 2 word the same note slightly differently.
PCT_SCALE_FOOTNOTE_MAIN = (
    "Effects are percentage changes in the OS hazard ratio over the stated increment of the "
    "exposure: 100 × (ratio − 1) for the exponentiated meta-regression coefficient (i.e. a value "
    "of +0.18% corresponds to a hazard ratio of 1.0018)."
)


# Column titles as printed.  The frames keep their original names; this is applied when a table is written.
def display_header(name):
    """Column title as printed in a published table."""
    if name == "p-value":
        return "P"
    if name == "95% CI (Hartung-Knapp)":
        return "95% CI"
    return name.replace("Tau²", "τ²").replace("tau²", "τ²")



def _pct_of_ratio(ratio, decimals=2):
    """1.0039 -> '+0.39%'."""
    return f"{(ratio - 1.0) * 100:+.{decimals}f}%".replace("-", "−")


def _fmt_p3(p_val):
    return "<0.001" if p_val < 0.001 else f"{p_val:.3f}"


# ── Random-effects meta-regression (REML tau^2, Hartung-Knapp, I^2 from the fixed-effect Q) ──
def meta_reg_hk(y, v, X, hk_floor=False):
    k, p = X.shape
    tau2 = 0.0
    for it in range(200):
        w = 1.0 / (v + tau2)
        W = np.diag(w)
        XtW = X.T @ W
        XtWX_inv = np.linalg.pinv(XtW @ X)
        P = W - W @ X @ XtWX_inv @ XtW
        PP = P @ P
        score = y.T @ PP @ y - np.trace(P)
        info = np.trace(PP)
        step = score / info
        tau2_new = max(0.0, tau2 + step)
        if abs(tau2_new - tau2) < 1e-10:
            tau2 = tau2_new
            break
        tau2 = tau2_new
    
    w = 1.0 / (v + tau2)
    W = np.diag(w)
    XtW = X.T @ W
    XtWX_inv = np.linalg.pinv(XtW @ X)
    beta = XtWX_inv @ XtW @ y
    r = y - X @ beta
    df_res = max(1, k - p)
    q = np.sum(w * r**2) / df_res
    if hk_floor: q = max(1.0, q)
    var_hk = q * XtWX_inv
    se_hk = np.sqrt(np.maximum(0, np.diag(var_hk)))
    t_stat = np.where(se_hk > 0, beta / se_hk, 0)
    p_hk = 2 * stats.t.sf(np.abs(t_stat), df=df_res)
    
    w_fixed = 1.0 / v
    W_f = np.diag(w_fixed)
    XtW_f = X.T @ W_f
    XtWX_f_inv = np.linalg.pinv(XtW_f @ X)
    beta_f = XtWX_f_inv @ XtW_f @ y
    r_f = y - X @ beta_f
    Q = np.sum(w_fixed * r_f**2)
    df_Q = k - p
    I2 = max(0.0, (Q - df_Q) / Q * 100.0) if Q > 0 else 0.0
    
    return {"k": k, "beta": beta, "se_hk": se_hk, "p_hk": p_hk, "tau2": tau2, "q": q, "I2": I2}


def _is_blank_row(values):
    return all(v is None or (isinstance(v, str) and not v.strip()) for v in values)


def read_sheet_block(ws, header_row=3):
    """Return (header, data_rows) for a sheet laid out as
    row 1 = caption, row 2 = blank, row 3 = header, row 4+ = data until a blank row."""
    header = [c.value for c in ws[header_row]]
    rows = []
    for values in ws.iter_rows(min_row=header_row + 1, values_only=True):
        if _is_blank_row(values):
            break
        rows.append(list(values))
    return header, rows


# ── Drug-class adjustment: reference and indicator levels ──
DRUG_CLASS_REFERENCE = "Immunotherapy (CPI-based)"
DRUG_CLASS_INDICATORS = ["Small-Molecule Targeted Therapy", "Other"]


# ── One random-effects fit of OS HR on an exposure (optionally PFS-adjusted) ──
def _fit_exposure(frame, x_col, adj_pfs, drop_impossible, x_scale=100.0, adj_class=False):
    """One random-effects meta-regression of log OS HR on an exposure.

    x_scale converts the stored exposure to percentage points (crossover is held
    as a fraction, geography already as a percentage).  drop_impossible applies
    the >100% rule that the progressed-denominator definitions carry.  adj_class
    appends two indicator columns for the grouped drug class (small-molecule
    targeted therapy; other), with checkpoint-inhibitor immunotherapy as the
    reference.  The exposure stays coefficient 1 whatever is appended after it, and
    the t quantile uses k minus the number of columns, so it follows the extra terms.
    """
    sub = frame if not drop_impossible else frame[~(frame[x_col] > 1.0)]
    need = ["log_os_hr", "se_log_hr", x_col] + (["log_pfs_hr"] if adj_pfs else [])
    valid = sub.dropna(subset=need)
    terms = [np.ones(len(valid)), valid[x_col].values * x_scale]
    if adj_pfs:
        terms.append(valid["log_pfs_hr"].values)
    if adj_class:
        assert valid["drug_class_grp"].isin([DRUG_CLASS_REFERENCE] + DRUG_CLASS_INDICATORS).all(), \
            "Unexpected grouped drug class"
        terms.extend((valid["drug_class_grp"] == _grp).astype(float).values for _grp in DRUG_CLASS_INDICATORS)
    X = np.column_stack(terms)
    res = meta_reg_hk(valid["log_os_hr"].values, valid["v"].values, X)
    b, se = res["beta"][1], res["se_hk"][1]
    t_crit = stats.t.ppf(0.975, df=res["k"] - X.shape[1])
    res.update(b=b, se_b=se, t_crit=t_crit, p1=res["p_hk"][1],
               ratio=np.exp(b * 10), lo=np.exp((b - t_crit * se) * 10),
               hi=np.exp((b + t_crit * se) * 10))
    return res


def _write_table(sheet_key, caption, frame, footnotes=(), group_headers=()):
    """group_headers: (label, first_col, last_col) 1-based spans written, merged, into
    row 2 above the column titles; row 3 stays the header row either way."""
    _ws = wb.create_sheet(title=SHEET_NAMES[sheet_key])
    _ws.append([caption])
    if group_headers:
        _band = [None] * len(frame.columns)
        for _label, _first, _last in group_headers:
            _band[_first - 1] = _label
        _ws.append(_band)
        for _label, _first, _last in group_headers:
            _ws.merge_cells(start_row=2, start_column=_first, end_row=2, end_column=_last)
    else:
        _ws.append([])
    _ws.append([display_header(c) if c != "" else None for c in frame.columns])  # "" = spacer column
    for _row in frame.itertuples(index=False):
        _ws.append(list(_row))
    if footnotes:
        _ws.append([])
        for _fn in footnotes:
            _ws.append([_fn])


# ── Main tables 1 and 2: one row per exposure definition ─────────────────────
# Slim summaries of the two full tables.  Every cell is assembled from the full-table
# rows and then checked against the full sheet as it was written, so the main tables
# cannot say anything the supplementary ones do not.
def _effect_cell(effect, ci):
    return f"{effect} ({ci})"


def _slim_rows(full_df, key_cols):
    """One slim row per key: k as 'unadjusted (PFS-adjusted)', plus effect and P for both."""
    _out = []
    for _key, _grp in full_df.groupby(key_cols, sort=False):
        _by_model = {_m: _g.iloc[0] for _m, _g in _grp.groupby("Model")}
        _u, _p = _by_model["Unadjusted"], _by_model["PFS-adjusted"]
        _row = dict(zip(key_cols, _key if isinstance(_key, tuple) else (_key,)))
        _row["Comparisons, k"] = f"{int(_u['Analyzed (k)'])} ({int(_p['Analyzed (k)'])})"
        _row["Unadjusted: OS HR change per +10 pp (95% CI)"] = _effect_cell(
            _u["OS HR change (per +10pp)"], _u["95% CI (Hartung-Knapp)"])
        _row["Unadjusted: P"] = _u["p-value"]
        _row["PFS-adjusted: OS HR change per +10 pp (95% CI)"] = _effect_cell(
            _p["OS HR change (per +10pp)"], _p["95% CI (Hartung-Knapp)"])
        _row["PFS-adjusted: P"] = _p["p-value"]
        _out.append(_row)
    return pd.DataFrame(_out)


def _check_slim_against_sheet(slim_df, sheet_key, key_cols):
    """Re-read the written full sheet and confirm every slim value against it."""
    _hdr, _rows = read_sheet_block(wb[SHEET_NAMES[sheet_key]])
    _full = pd.DataFrame(_rows, columns=_hdr)
    _full = _full.rename(columns={display_header(_c): _c for _c in ("95% CI (Hartung-Knapp)", "p-value")})
    assert len(_full) == 3 * len(slim_df), \
        f"{TABLE_LABEL[sheet_key]} should carry 3 model rows per slim row, has {len(_full)} for {len(slim_df)}"
    for _, _s in slim_df.iterrows():
        _m = pd.Series(True, index=_full.index)
        for _c in key_cols:
            _m &= _full[_c] == _s[_c]
        _g = _full[_m].set_index("Model")
        assert len(_g) == 3, f"{TABLE_LABEL[sheet_key]}: expected 3 model rows for {dict(_s[key_cols])}"
        _u, _p = _g.loc["Unadjusted"], _g.loc["PFS-adjusted"]
        assert _s["Comparisons, k"] == f"{_u['Analyzed (k)']} ({_p['Analyzed (k)']})"
        assert _s["Unadjusted: OS HR change per +10 pp (95% CI)"] == \
            f"{_u['OS HR change (per +10pp)']} ({_u['95% CI (Hartung-Knapp)']})"
        assert _s["Unadjusted: P"] == _u["p-value"]
        assert _s["PFS-adjusted: OS HR change per +10 pp (95% CI)"] == \
            f"{_p['OS HR change (per +10pp)']} ({_p['95% CI (Hartung-Knapp)']})"
        assert _s["PFS-adjusted: P"] == _p["p-value"]


def run_regression(data, x_col, y_col):
    sub = data[[x_col, y_col]].dropna()
    n = len(sub)
    if n < 3:
        return {"n": n, "r": np.nan, "r2": np.nan, "slope": np.nan, "p": np.nan}
    slope, intercept, r_val, p_val, _ = stats.linregress(sub[x_col], sub[y_col])
    return {"n": n, "r": r_val, "r2": r_val ** 2, "slope": slope, "p": p_val}


def build_collapsed_df(base_df, arm_split_drop_regimen, target_split_drop_specific):
    drop_mask = pd.Series(False, index=base_df.index)
    for study, drop_regimen in arm_split_drop_regimen.items():
        m = (base_df["_study_stripped"] == study) & (
            base_df["regimen"].astype(str).str.strip() == drop_regimen
        )
        assert m.sum() == 1, f"Expected exactly 1 row to drop for {study} ({drop_regimen}), found {m.sum()}"
        drop_mask |= m
    for study, drop_specific in target_split_drop_specific.items():
        m = (base_df["_study_stripped"] == study) & (
            base_df["Specific Treatment"].astype(str).str.strip() == drop_specific
        )
        assert m.sum() == 1, f"Expected exactly 1 row to drop for {study} ({drop_specific}), found {m.sum()}"
        drop_mask |= m
    return base_df[~drop_mask].copy()


def build_dedup_geo_df(base_df, target_split_drop_specific):
    """One row per comparison for the geography-vs-OS models.

    Western enrollment is a trial-level constant, identical across every row of
    the same trial.  For the target-split pairs the OS hazard ratio is identical
    too, because both rows describe the same comparison -- they differ only in
    which drug of the experimental regimen carried the qualifying prior
    later-line approval.  In a model that does not involve crossover those two
    rows are the same observation entered twice, receiving double
    inverse-variance weight, so the second one is dropped here.

    The arm-split pairs are kept: same Western enrollment, but genuinely
    different OS hazard ratios against a shared control, so they are distinct
    observations.

    Reuses TARGET_SPLIT_DROPS rather than a second hardcoded list, so the
    geography dedup and the collapsed sensitivity analysis can never drift apart.
    """
    study_col = base_df["study"].astype(str).str.strip()
    specific_col = base_df["Specific Treatment"].astype(str).str.strip()
    drop_mask = pd.Series(False, index=base_df.index)
    for study, drop_specific in target_split_drop_specific.items():
        m = (study_col == study) & (specific_col == drop_specific)
        assert m.sum() == 1, (
            f"Expected exactly 1 duplicated geography row to drop for "
            f"{study} ({drop_specific}), found {m.sum()}"
        )
        drop_mask |= m
    assert int(drop_mask.sum()) == len(target_split_drop_specific), (
        f"Expected to drop exactly {len(target_split_drop_specific)} rows from the "
        f"geography dataset, dropped {int(drop_mask.sum())}"
    )
    return base_df[~drop_mask].copy()

## 4. Data loading and derived variables
`raw_df` is the analysis frame: one row per comparison, with the derived columns (log OS HR and its standard
error, drug class, clinical setting, trial size, start era) that the tables use. Excel errors in the working
master were written as "NR" by `scripts/03_make_analysis_dataset.py`, so "NR" and blank both read as missing.

In [4]:
raw_df = pd.read_excel(MASTER_FILE, sheet_name=SHEET)

for col in ["perc_pts_NorthAmerica_Europe", "perc_pts_Europe", "perc_pts_WesternEurope", "perc_pts_US", "perc_pts_NorthAmerica"]:
    if col in raw_df.columns:
        raw_df[col] = raw_df[col].apply(parse_geo_pct)

raw_df["_max_europe"] = raw_df[["perc_pts_Europe", "perc_pts_WesternEurope"]].max(axis=1)
raw_df["_max_us"]     = raw_df[["perc_pts_US", "perc_pts_NorthAmerica"]].max(axis=1)

raw_df["log_os_hr"]   = np.log(pd.to_numeric(raw_df["Combined_OS_HR"], errors="coerce"))

raw_df["_drug_class"] = raw_df["Specific Treatment"].apply(classify_drug_class_from_specific)

raw_df["clinical_setting"] = raw_df["context"].apply(group_context)
raw_df["ITT_Control_Num"] = pd.to_numeric(raw_df["ITT_Control_Num"], errors="coerce")
raw_df["trial_size"] = np.where(raw_df["ITT_Control_Num"] < 200, "Small (<200)",
                        np.where(raw_df["ITT_Control_Num"] < 500, "Medium (200–499)", "Large (≥500)"))
raw_df["start_year"] = raw_df.apply(get_start_year, axis=1)
raw_df["start_era"] = np.where(raw_df["start_year"] < 2015, "Pre-2015", "2015+")
raw_df["drug_class_grp"] = raw_df["_drug_class"].apply(group_drug_class_for_regression)

cross_cols = [col_exact_sub, col_inclass_sub, col_exact_prog, col_inclass_prog, col_exact_event, col_inclass_event]
for c in cross_cols:
    raw_df[c] = pd.to_numeric(raw_df[c], errors="coerce")

# Pre-computation for the meta-regressions: log hazard ratios and the standard error of log OS HR
# Pre-computation safety for log HR metrics
for d in [raw_df]:
    if "se_log_hr" not in d.columns:
        d["Combined_OS_HR_CI_low"] = pd.to_numeric(d["Combined_OS_HR_CI_low"], errors="coerce")
        d["Combined_OS_HR_CI_high"] = pd.to_numeric(d["Combined_OS_HR_CI_high"], errors="coerce")
        d["se_log_hr"] = (np.log(d["Combined_OS_HR_CI_high"]) - np.log(d["Combined_OS_HR_CI_low"])) / (2 * 1.959964)
        d["v"] = d["se_log_hr"]**2
        d["log_os_hr"] = np.log(pd.to_numeric(d["Combined_OS_HR"], errors="coerce"))
        d["log_pfs_hr"] = np.log(pd.to_numeric(d["Combined_PFS_HR"], errors="coerce"))

# Precision guard.  A standard error of log OS HR above 1.5 would mean a 95% confidence interval spanning
# roughly 0.05x to 19x the point estimate -- not an estimate worth weighting.  No comparison comes close
# (the widest is 0.46), so this is asserted rather than filtered: a filter that removes nothing reads as
# though it were doing work, and would drop comparisons silently if the data changed.
_SE_LOG_HR_MAX = 1.5
assert (raw_df["se_log_hr"].dropna() <= _SE_LOG_HR_MAX).all(), (
    f"Standard error of log OS HR exceeds {_SE_LOG_HR_MAX} for "
    f"{int((raw_df['se_log_hr'] > _SE_LOG_HR_MAX).sum())} comparison(s) -- decide "
    f"explicitly how to handle them rather than dropping them silently"
)
df_meta = raw_df.copy()
if "_drug_class" not in df_meta.columns:
    df_meta["_drug_class"] = df_meta["Specific Treatment"].apply(classify_drug_class_from_specific)
df_meta["_study_stripped"] = df_meta["study"].astype(str).str.strip()

# ── Drift guard: the unweighted primary correlations of the full 86-comparison dataset ──
CROSS_COL = col_inclass_sub
GEO_COL = "perc_pts_NorthAmerica_Europe"
OS_COL = "log_os_hr"
primary_a1 = run_regression(raw_df, CROSS_COL, OS_COL)
primary_a2 = run_regression(raw_df, GEO_COL, CROSS_COL)

# Drift guards, pinned to the current analysis dataset (78 trials, 86 comparisons).  BASELINE_A1_P is the
# in-class definition (CROSS_COL), which is what this unweighted check evaluates; the base-case
# meta-regression uses exact-drug crossover with the subsequent-therapy denominator.
BASELINE_A1_P = 0.487
BASELINE_A2_P = 0.999

print("Baseline check -- Analysis 1 (Crossover -> log OS):", primary_a1, f"expected p ~ {BASELINE_A1_P}")
print("Baseline check -- Analysis 2 (Geography -> Crossover):", primary_a2, f"expected p ~ {BASELINE_A2_P}")

assert abs(primary_a1["p"] - BASELINE_A1_P) < 0.01, "Analysis 1 baseline does not match manuscript -- stop."
assert abs(primary_a2["p"] - BASELINE_A2_P) < 0.01, "Analysis 2 baseline does not match manuscript -- stop."

assert len(build_collapsed_df(df_meta, ARM_SPLIT_DROPS, TARGET_SPLIT_DROPS)) == 78, "collapsing arm- and target-split rows should leave 78 trials"

# ── Comparisons whose progressed-denominator crossover exceeds 100% (Supplementary Table S1) ──
# Manuscript column order for the impossible-crossover table.
IMPOSSIBLE_COL_ORDER = [
    "Study", "Regimen", "Numerator (Exact)", "Numerator (In-Class)",
    "Denominator (Progressed)", "Crossover Rate", "Reason",
]

IMP_NUM_EXACT   = "Num control pts getting exact drug post-protocol"
IMP_NUM_INCLASS = "Num control pts getting post-protocol drug in class"
IMP_DENOM       = "Total_Control_Pts_Progressed"
for _c in [IMP_NUM_EXACT, IMP_NUM_INCLASS, IMP_DENOM]:
    raw_df[_c] = pd.to_numeric(raw_df[_c], errors="coerce")

_over_exact   = raw_df[col_exact_prog] > 1.0
_over_inclass = raw_df[col_inclass_prog] > 1.0

# The Table 1 footnote quotes both counts from these variables, so the footnote and the table cannot disagree.
n_over_exact   = int(_over_exact.sum())
n_over_inclass = int(_over_inclass.sum())
assert n_over_exact == EXPECTED["n_over_exact"], \
    f'Expected {EXPECTED["n_over_exact"]} exact-drug comparisons >100%, got {n_over_exact}'
assert n_over_inclass == EXPECTED["n_over_inclass"], \
    f'Expected {EXPECTED["n_over_inclass"]} in-class comparisons >100%, got {n_over_inclass}'

_studies_exact   = set(raw_df.loc[_over_exact, "study"].astype(str).str.strip())
_studies_inclass = set(raw_df.loc[_over_inclass, "study"].astype(str).str.strip())
assert _studies_exact <= _studies_inclass, (
    "Footnote arithmetic assumes every exact-drug violator also violates in-class; "
    f"exceptions: {sorted(_studies_exact - _studies_inclass)}"
)

_imp_src = raw_df[_over_exact | _over_inclass].copy()
_imp_src["_viol_exact"]   = _over_exact[_imp_src.index]
_imp_src["_viol_inclass"] = _over_inclass[_imp_src.index]
# exact-drug violators first (as the table has always been laid out), then the
# in-class-only ones; alphabetical within each group for a stable order.
_imp_src = _imp_src.sort_values(
    by=["_viol_exact", "study"],
    key=lambda col: col.map(lambda v: not v) if col.name == "_viol_exact" else col.astype(str).str.lower(),
)

def _int_or_none(v):
    return None if pd.isna(v) else int(v)

impossible_rows = []
for _, _r in _imp_src.iterrows():
    _nx, _nc, _d = _r[IMP_NUM_EXACT], _r[IMP_NUM_INCLASS], _r[IMP_DENOM]
    _rx, _rc = _r[col_exact_prog], _r[col_inclass_prog]
    if _r["_viol_exact"] and _r["_viol_inclass"]:
        _reason = (f"Exact-drug numerator ({_int_or_none(_nx)}) and in-class numerator "
                   f"({_int_or_none(_nc)}) both exceed control patients progressed ({_int_or_none(_d)})")
    elif _r["_viol_inclass"]:
        _reason = (f"In-class numerator ({_int_or_none(_nc)}) exceeds control patients "
                   f"progressed ({_int_or_none(_d)})")
    else:
        _reason = (f"Exact-drug numerator ({_int_or_none(_nx)}) exceeds control patients "
                   f"progressed ({_int_or_none(_d)})")
    _rate = max([v for v in (_rx if _r["_viol_exact"] else None,
                             _rc if _r["_viol_inclass"] else None) if v is not None])
    impossible_rows.append({
        "Study": str(_r["study"]).strip(),
        "Regimen": str(_r["regimen"]).strip(),
        "Numerator (Exact)": _int_or_none(_nx),
        "Numerator (In-Class)": _int_or_none(_nc),
        "Denominator (Progressed)": _int_or_none(_d),
        "Crossover Rate": f"{_rate*100:.1f}%",
        "Reason": _reason,
    })

df_impossible = pd.DataFrame(impossible_rows, columns=IMPOSSIBLE_COL_ORDER)
# The experimental drug that carries the qualifying approval, row for row, to be set in bold in multi-drug regimens.
IMPOSSIBLE_PRIMARY_DRUG = list(_imp_src["Specific Treatment"].astype(str).str.strip())
assert len(df_impossible) == EXPECTED["n_impossible_rows"], \
    f'Expected {EXPECTED["n_impossible_rows"]} impossible-crossover comparisons, got {len(df_impossible)}'

Baseline check -- Analysis 1 (Crossover -> log OS): {'n': 72, 'r': np.float64(0.08320476705231931), 'r2': np.float64(0.006923033260230722), 'slope': np.float64(0.06688092532556443), 'p': np.float64(0.48713937374308014)} expected p ~ 0.487
Baseline check -- Analysis 2 (Geography -> Crossover): {'n': 46, 'r': np.float64(-0.00010713412587264677), 'r2': np.float64(1.1477720926496123e-08), 'slope': np.float64(-8.930282792765219e-07), 'p': np.float64(0.9994361977232344)} expected p ~ 0.999


## 5. Table 1 and Supplementary Tables S4-S5
Random-effects meta-regression of OS hazard ratio on control-arm crossover, for the six numerator x
denominator definitions, each unadjusted and PFS-adjusted. S4 holds every model, S5 the between-trial
heterogeneity explained by crossover and PFS HR, and Table 1 the unadjusted and PFS-adjusted estimates for
each definition. (Supplementary Table S6 combines these fits with the Western-enrollment fits, so it is
written in section 6.)

In [5]:
# numerator x denominator grid shared by the full crossover table and the precision-bounds table
XOVER_SPEC = [
    ("Primary",               "exact-drug", "subsequent therapy", col_exact_sub,   False),
    ("Principal sensitivity", "in-class",   "subsequent therapy", col_inclass_sub, False),
    ("Sensitivity",           "exact-drug", "progressed",         col_exact_prog,  True),
    ("Sensitivity",           "in-class",   "progressed",         col_inclass_prog, True),
    ("Sensitivity",           "exact-drug", "PFS/EFS event",      col_exact_event,  False),
    ("Sensitivity",           "in-class",   "PFS/EFS event",      col_inclass_event, False),
]

# (model label, adjust for log PFS HR, adjust for drug class)
FULL_MODELS = [
    ("Unadjusted",          False, False),
    ("PFS-adjusted",        True,  False),
    ("Drug-class-adjusted", False, True),
]

# The precision-bounds table scales a fit to this between-trial contrast.
BOUNDS_PP = 50
BOUNDS_CONTRAST_COL = f"OS HR change ({BOUNDS_PP}-pp contrast), 95% CI"
BOUNDS_COLS = ["Role", "Numerator", "Denominator", "Model", "Analyzed (k)",
               "OS HR change (per +10pp), 95% CI", BOUNDS_CONTRAST_COL,
               "Largest OS-HR Increase Compatible", "p-value"]


def _bounds_row(role, numerator, denominator, model, f):
    """One precision-bounds row from a fit, scaled to a BOUNDS_PP-point contrast."""
    _b, _se, _tc = f["b"], f["se_b"], f["t_crit"]
    _c, _clo, _chi = (np.exp(_b * BOUNDS_PP), np.exp((_b - _tc * _se) * BOUNDS_PP),
                      np.exp((_b + _tc * _se) * BOUNDS_PP))
    return {
        "Role": role, "Numerator": numerator, "Denominator": denominator, "Model": model,
        "Analyzed (k)": int(f["k"]),
        "OS HR change (per +10pp), 95% CI":
            f"{_pct_of_ratio(f['ratio'])} ({_pct_of_ratio(f['lo'])} to {_pct_of_ratio(f['hi'])})",
        BOUNDS_CONTRAST_COL: f"{_pct_of_ratio(_c, 1)} ({_pct_of_ratio(_clo, 1)} to {_pct_of_ratio(_chi, 1)})",
        "Largest OS-HR Increase Compatible": _pct_of_ratio(_chi, 1),
        "p-value": _fmt_p3(f["p1"]),
    }

xover_full_rows, bounds_rows = [], []
for _role, _num, _den, _col, _drop_imp in XOVER_SPEC:
    _avail = int((df_meta["log_os_hr"].notna() & df_meta[_col].notna()).sum())
    _ols = df_meta[["log_os_hr", _col]].dropna()
    if _drop_imp:
        _ols = _ols[_ols[_col] <= 1.0]
    _r_ols, _p_ols = stats.pearsonr(_ols[_col], _ols["log_os_hr"])
    for _model, _adj, _cls in FULL_MODELS:
        _f = _fit_exposure(df_meta, _col, _adj, _drop_imp, adj_class=_cls)
        xover_full_rows.append({
            "Role": _role, "Numerator": _num, "Denominator": _den, "Model": _model,
            "Comparisons available": _avail,
            "Comparisons excluded": _avail - int(_f["k"]),
            "Analyzed (k)": int(_f["k"]),
            "OS HR change (per +10pp)": _pct_of_ratio(_f["ratio"]),
            "95% CI (Hartung-Knapp)": f"{_pct_of_ratio(_f['lo'])} to {_pct_of_ratio(_f['hi'])}",
            "p-value": _fmt_p3(_f["p1"]), "tau²": f"{_f['tau2']:.5f}",
            "I² (%)": f"{_f['I2']:.1f}",
            "OLS r (unweighted, descriptive)": f"{_r_ols:+.3f}".replace("-", "−"),
            "OLS n": int(len(_ols)),
            "OLS p (unweighted, descriptive)": _fmt_p3(_p_ols),
        })
        # The precision-bounds table scales the unadjusted and PFS-adjusted fits only.
        if not _cls:
            bounds_rows.append(_bounds_row(_role, _num, _den, _model, _f))

df_xover_full = pd.DataFrame(xover_full_rows)
# denominator for the ">100% crossover" rate quoted in the manuscript text
_inclass_prog_available = int(df_xover_full.loc[(df_xover_full["Numerator"] == "in-class")
                                        & (df_xover_full["Denominator"] == "progressed"),
                                        "Comparisons available"].iloc[0])
# 6 definitions x 3 models for the full table; the bounds table takes the first two models only
# here, and gains the Western-enrollment rows once the geography fits exist (below).
assert len(df_xover_full) == 18 and len(bounds_rows) == 12, \
    "The full crossover table must carry 18 model rows and the crossover bounds 12"
assert list(df_xover_full["Model"].iloc[:3]) == [m for m, _, _ in FULL_MODELS]
# The PCT_SCALE_FOOTNOTE worked example is the primary unadjusted effect; keep it in step.
assert df_xover_full.iloc[0]["OS HR change (per +10pp)"] == "+0.18%", \
    "PCT_SCALE_FOOTNOTE quotes +0.18% / 1.0018 as its worked example; the primary unadjusted effect has moved"

# Heterogeneity table: how much between-trial heterogeneity each covariate explains, on the
# comparisons where OS, its standard error, the crossover rate and the PFS
# hazard ratio are all present, so the four models are nested on one sample.
hetero_rows = []
for _lab, _col in (("exact-drug", col_exact_sub), ("in-class", col_inclass_sub)):
    _valid = df_meta.dropna(subset=["log_os_hr", "se_log_hr", "log_pfs_hr", _col])
    _y, _v, _n = _valid["log_os_hr"].values, _valid["v"].values, len(_valid)
    _x, _pfs = _valid[_col].values * 100.0, _valid["log_pfs_hr"].values
    _t_null = meta_reg_hk(_y, _v, np.ones((_n, 1)))["tau2"]
    _t_x    = meta_reg_hk(_y, _v, np.column_stack([np.ones(_n), _x]))["tau2"]
    _t_pfs  = meta_reg_hk(_y, _v, np.column_stack([np.ones(_n), _pfs]))["tau2"]
    _t_full = meta_reg_hk(_y, _v, np.column_stack([np.ones(_n), _x, _pfs]))["tau2"]
    _r2 = lambda t: f"{max(0.0, (_t_null - t) / _t_null * 100.0):.1f}%" if _t_null > 0 else "0.0%"
    hetero_rows.append({
        "Crossover Definition": _lab, "k": _n, "Null Tau²": f"{_t_null:.5f}",
        "Tau² with Crossover Alone": f"{_t_x:.5f}", "R²_meta (Crossover Alone)": _r2(_t_x),
        "Tau² with PFS HR Alone": f"{_t_pfs:.5f}", "R²_meta (PFS HR Alone)": _r2(_t_pfs),
        "Tau² Full Model (Crossover + PFS)": f"{_t_full:.5f}",
        "R²_meta (Full Model)": _r2(_t_full),
    })
df_hetero = pd.DataFrame(hetero_rows)

In [6]:
XOVER_FULL_FOOTNOTE = (
    "Comparisons excluded for one or more of the following reasons: crossover rate exceeding "
    "100% of the stated denominator, indicating an internally inconsistent report "
    f"(progressed-denominator definitions only; {n_over_exact} exact-drug and {n_over_inclass} "
    "in-class comparisons); an overall survival hazard ratio reported without a confidence "
    "interval, from which no standard error could be derived (1 comparison, IFM 2005-02); or no "
    "reported PFS hazard ratio (1 comparison, IMvigor130; PFS-adjusted models only). "
    f"Comparisons with crossover rates above 100% are listed in {TABLE_LABEL['S_IMPOSSIBLE']}."
)
BOUNDS_FOOTNOTE = (
    "Bounds are the exponentiated limits of the 95% Hartung-Knapp confidence interval for the "
    f"crossover coefficient, scaled to a {BOUNDS_PP}-percentage-point between-trial contrast. They describe "
    "variation across trials and do not estimate the effect of increasing crossover within a "
    f"single trial. For Western enrollment, the contrast is {BOUNDS_PP} percentage points (pp) of participants "
    "enrolled in North America or Europe."
)
DRUG_CLASS_FOOTNOTE = (
    "Drug-class-adjusted models include indicators for small-molecule targeted therapy and other "
    "drug classes, with checkpoint inhibitors as the reference."
)

In [7]:
_write_table("S_XOVER_FULL", f"{TABLE_LABEL['S_XOVER_FULL']}. Crossover vs log overall survival hazard ratio, all models.",
             df_xover_full, (XOVER_FULL_FOOTNOTE, DRUG_CLASS_FOOTNOTE, PCT_SCALE_FOOTNOTE))
_write_table("S_HETERO", f"{TABLE_LABEL['S_HETERO']}. Between-trial heterogeneity for OS HR explained by crossover and PFS HR.",
             df_hetero)
print(f"Built {TABLE_LABEL['S_XOVER_FULL']} ({len(df_xover_full)} rows) and {TABLE_LABEL['S_HETERO']} ({len(df_hetero)} rows); "
      f"primary crossover k={df_xover_full.iloc[0]['Analyzed (k)']}.")

Built Supplementary Table S4 (18 rows) and Supplementary Table S5 (2 rows); primary crossover k=54.


In [8]:
# Main crossover table: one row per crossover definition
df_t_xover = _slim_rows(df_xover_full, ["Numerator", "Denominator"])
assert len(df_t_xover) == 6, f"Expected 6 rows for {TABLE_LABEL['T_XOVER']}, got {len(df_t_xover)}"
_check_slim_against_sheet(df_t_xover, "S_XOVER_FULL", ["Numerator", "Denominator"])

T_XOVER_CAPTION = (f"{TABLE_LABEL['T_XOVER']}. Association between control-arm crossover and log overall "
                   "survival hazard ratio, by crossover definition.")
T_XOVER_FN_DEFS = (
    "Numerator: exact-drug = control patients who subsequently received the experimental drug; "
    "in-class = control patients who received any drug in the same class as the experimental drug. "
    "Denominator: subsequent therapy = control patients who received any subsequent systemic "
    "therapy (primary definition); progressed = control patients with progression; PFS/EFS event = "
    "control patients with a PFS, DFS or EFS event."
)
T_XOVER_FN_OVER100 = (
    f"Progressed-denominator crossover rates above 100% are excluded ({n_over_exact} exact-drug "
    f"and {n_over_inclass} in-class comparisons; {TABLE_LABEL['S_IMPOSSIBLE']}); results with these "
    f"rates capped at 100% are in {TABLE_LABEL['S_CAPPED']}."
)
PFS_K_FOOTNOTE = ("PFS-adjusted models additionally include log PFS HR; k in parentheses excludes "
                  "comparisons without a reported PFS HR.")
T_XOVER_FN_ABBREV = 'Abbreviations: pp: percentage-points; PFS: Progression-free survival; EFS: Event-free survival; OS: Overall survival; HR: Hazard ratio; CI: Confidence interval.'
T_XOVER_FN_FULL = (
    "Full results, including heterogeneity, drug-class-adjusted models and unweighted correlations, "
    f"are in {TABLE_LABEL['S_XOVER_FULL']}."
)
_write_table("T_XOVER", T_XOVER_CAPTION, df_t_xover,
             (T_XOVER_FN_DEFS, T_XOVER_FN_OVER100, PFS_K_FOOTNOTE, T_XOVER_FN_FULL, PCT_SCALE_FOOTNOTE_MAIN,
              T_XOVER_FN_ABBREV))
print(f"Built {TABLE_LABEL['T_XOVER']} ({len(df_t_xover)} rows).")

Built Table 1 (6 rows).


## 6. Table 2 and Supplementary Tables S6-S7
The same meta-regression with Western enrollment (North America plus Europe) as the exposure. Target-split
trials are entered once here, because crossover plays no part in this model and the two rows are the same
observation. S7 holds all geography models and Table 2 the unadjusted and PFS-adjusted Western-enrollment
estimates. S6 (precision bounds) combines the crossover fits from section 5 with these fits, which is why it
is written here rather than with Table 1.

In [9]:
# ── Full geography table: Western enrollment vs log OS hazard ratio ──────────
# Western enrollment is a trial-level constant, and target-split rows duplicate the same
# comparison (identical OS HR and geography), so they are dropped from geography models.
# Arm-split rows are retained: same geography, but genuinely different OS HRs.
df_geo = build_dedup_geo_df(df_meta, TARGET_SPLIT_DROPS)

_geo_dropped = df_meta.loc[df_meta.index.difference(df_geo.index)]
print(f"\nGeography dataset: dropped the {len(_geo_dropped)} target-split duplicate rows "
      f"({len(df_meta)} -> {len(df_geo)} comparisons)")
for _, _r in _geo_dropped.iterrows():
    print(f"  - {str(_r['study']).strip()} ({str(_r['Specific Treatment']).strip()} arm): "
          f"regimen={str(_r['regimen']).strip()}; OS HR={_r['Combined_OS_HR']}; "
          f"Western enrollment={_r['perc_pts_NorthAmerica_Europe']}")

WESTERN_LABEL = "Western enrollment (North America + Europe)"
GEO_METRICS = [
    ("Primary",     WESTERN_LABEL,               "perc_pts_NorthAmerica_Europe"),
    ("Sensitivity", "Europe or Western Europe",  "_max_europe"),
    ("Sensitivity", "US or North America",       "_max_us"),
]


geo_full_rows = []
for _role, _geo_label, _geo_col in GEO_METRICS:
    assert _geo_col in df_geo.columns, f"The geography table needs the parsed column {_geo_col}"
    _eligible = df_geo["log_os_hr"].notna() & df_geo[_geo_col].notna()
    _ols = df_geo.loc[_eligible, [_geo_col, "log_os_hr"]]
    _ols_r, _ols_p = stats.pearsonr(_ols[_geo_col], _ols["log_os_hr"])
    for _model, _adj_pfs, _cls in FULL_MODELS:
        # Geography is already held as a percentage, so x_scale is 1.
        _f = _fit_exposure(df_geo, _geo_col, _adj_pfs, False, x_scale=1.0, adj_class=_cls)
        geo_full_rows.append({
            "Role": _role,
            "Geographic Metric": _geo_label,
            "Model": _model,
            "Comparisons available": int(_eligible.sum()),
            "Comparisons excluded": int(_eligible.sum()) - int(_f["k"]),
            "Analyzed (k)": int(_f["k"]),
            "OS HR change (per +10pp)": _pct_of_ratio(_f["ratio"]),
            "95% CI (Hartung-Knapp)": f"{_pct_of_ratio(_f['lo'])} to {_pct_of_ratio(_f['hi'])}",
            "p-value": _fmt_p3(_f["p1"]),
            "tau²": f"{_f['tau2']:.5f}",
            "I² (%)": f"{_f['I2']:.1f}",
            "OLS r (unweighted, descriptive)": f"{_ols_r:+.3f}".replace("-", "−"),
            "OLS n": int(len(_ols)),
            "OLS p (unweighted, descriptive)": _fmt_p3(_ols_p),
        })
        # The primary metric's unadjusted and PFS-adjusted fits join the precision-bounds table.
        if _role == "Primary" and not _cls:
            bounds_rows.append(_bounds_row("Geography", WESTERN_LABEL, "-", _model, _f))

df_geo_full = pd.DataFrame(geo_full_rows)
assert len(df_geo_full) == 9, f"Expected 9 rows for {TABLE_LABEL['S_GEO_FULL']}, got {len(df_geo_full)}"
print(f"\n{TABLE_LABEL['S_GEO_FULL']} recomputed on the deduplicated geography dataset:")
print(df_geo_full[["Geographic Metric", "Model", "Comparisons available", "Comparisons excluded",
             "Analyzed (k)", "OS HR change (per +10pp)", "95% CI (Hartung-Knapp)",
             "p-value"]].to_string(index=False))

GEO_FULL_CAPTION = (f"{TABLE_LABEL['S_GEO_FULL']}. Western enrollment vs log overall survival hazard ratio, "
              "all models. Geography "
              "analyses include one comparison per trial, except where a trial contributed "
              "two distinct experimental arms.")
GEO_FULL_FOOTNOTE = (
    "Comparisons excluded for one or more of the following reasons: an overall survival hazard "
    "ratio reported without a confidence interval, from which no standard error could be derived "
    "(1 comparison, IFM 2005-02); or no reported PFS hazard ratio (1 comparison, IMvigor130; "
    "PFS-adjusted models only)."
)

ws_geo_full = wb.create_sheet(title=SHEET_NAMES["S_GEO_FULL"])
ws_geo_full.append([GEO_FULL_CAPTION])
ws_geo_full.append([])
ws_geo_full.append([display_header(_c) for _c in df_geo_full.columns])
for _r in df_geo_full.itertuples(index=False):
    ws_geo_full.append(list(_r))
ws_geo_full.append([])
ws_geo_full.append([GEO_FULL_FOOTNOTE])
ws_geo_full.append([DRUG_CLASS_FOOTNOTE])
ws_geo_full.append([PCT_SCALE_FOOTNOTE])


Geography dataset: dropped the 4 target-split duplicate rows (86 -> 82 comparisons)
  - CLEAR (Lenvatinib arm): regimen=Lenvatinib and Pembrolizumab; OS HR=0.79; Western enrollment=55.800000000000004
  - COMBI-AD (Trametinib arm): regimen=Dabrafenib and Trametinib; OS HR=0.8; Western enrollment=85.7
  - COMBI-v (Trametinib arm): regimen=Dabrafenib and Trametinib; OS HR=0.69; Western enrollment=nan
  - RENOTORCH (Axitinib arm): regimen=Axitinib and Toripalimab; OS HR=0.61; Western enrollment=0.0

Supplementary Table S7 recomputed on the deduplicated geography dataset:
                          Geographic Metric               Model  Comparisons available  Comparisons excluded  Analyzed (k) OS HR change (per +10pp) 95% CI (Hartung-Knapp) p-value
Western enrollment (North America + Europe)          Unadjusted                     49                     0            49                   −1.63%       −3.72% to +0.49%   0.128
Western enrollment (North America + Europe)        PFS-adjusted    

In [10]:
# ── Precision bounds: crossover rows (above) plus the Western-enrollment rows ─
df_bounds = pd.DataFrame(bounds_rows, columns=BOUNDS_COLS)
assert len(df_bounds) == 14, f"Expected 14 rows for {TABLE_LABEL['S_BOUNDS']}, got {len(df_bounds)}"
# The compatible-increase column is the contrast upper limit, restated.
for _r in bounds_rows:
    assert _r["Largest OS-HR Increase Compatible"] == \
        _r[BOUNDS_CONTRAST_COL].split("(")[1].rstrip(")").split(" to ")[1], \
        f"{TABLE_LABEL['S_BOUNDS']} compatible-increase column disagrees with the contrast upper limit"
_write_table("S_BOUNDS", f"{TABLE_LABEL['S_BOUNDS']}. Nested random-effects meta-regression to estimate the largest "
                   "crossover or Western enrollment effect compatible with the observed data.",
             df_bounds, (BOUNDS_FOOTNOTE, PCT_SCALE_FOOTNOTE))
print(f"Built {TABLE_LABEL['S_BOUNDS']} ({len(df_bounds)} rows, of which {len(df_bounds) - 12} Western enrollment).")

Built Supplementary Table S6 (14 rows, of which 2 Western enrollment).


In [11]:
# Main geography table: one row per geographic metric
df_t_geo = _slim_rows(df_geo_full, ["Geographic Metric"])
assert len(df_t_geo) == 3, f"Expected 3 rows for {TABLE_LABEL['T_GEO']}, got {len(df_t_geo)}"
_check_slim_against_sheet(df_t_geo, "S_GEO_FULL", ["Geographic Metric"])

# Trials whose enrollment was taken from the closest reported geographic grouping.
GEO_PROXY_TRIALS = ["ECHELON-1", "KEYNOTE-811", "KEYNOTE-859"]
_studies_in_data = set(raw_df["study"].astype(str).str.strip())
assert set(GEO_PROXY_TRIALS) <= _studies_in_data, \
    f"Proxy-grouping footnote names trials not in the analysis set: {sorted(set(GEO_PROXY_TRIALS) - _studies_in_data)}"
T_GEO_CAPTION = (f"{TABLE_LABEL['T_GEO']}. Association between Western enrollment and log overall survival "
                 "hazard ratio, by geographic definition.")
T_GEO_FN_DEFS = (
    "Western enrollment = proportion of participants enrolled in North America or Europe; secondary "
    "metrics = North America (or US) and Europe (or Western Europe)."
)
T_GEO_FN_PROXY = (
    f"Enrollment was taken from the closest reported grouping for {len(GEO_PROXY_TRIALS)} trials: "
    "ECHELON-1 reported the Americas (including Latin America); KEYNOTE-811 and KEYNOTE-859 "
    "reported groupings that included Israel and Australia."
)
T_GEO_FN_ABBREV = 'Abbreviations: pp: Percentage-points; OS: Overall survival; HR: Hazard ratio; PFS: Progression-free survival.'
T_GEO_FN_FULL = (
    "Full results, including drug-class-adjusted models and unweighted correlations, are in "
    f"{TABLE_LABEL['S_GEO_FULL']}."
)
_write_table("T_GEO", T_GEO_CAPTION, df_t_geo,
             (T_GEO_FN_DEFS, T_GEO_FN_PROXY, PFS_K_FOOTNOTE, T_GEO_FN_FULL, PCT_SCALE_FOOTNOTE_MAIN,
              T_GEO_FN_ABBREV))
print(f"Built {TABLE_LABEL['T_XOVER']} ({len(df_t_xover)} rows) and {TABLE_LABEL['T_GEO']} ({len(df_t_geo)} rows).")

Built Table 1 (6 rows) and Table 2 (3 rows).


## 7. Supplementary Table S8 - all stratified correlations
Descriptive Pearson correlations (ordinary least squares) within 11 strata, for crossover vs log OS HR,
Western enrollment vs log OS HR and Western enrollment vs crossover, with Benjamini-Hochberg adjustment
across every computable correlation.

In [12]:
STRATIFICATIONS_11 = [
    ("Overall", "overall", ["overall"]),
    ("Clinical Setting", "clinical_setting", ["First-line (Metastatic/Advanced)", "Adjuvant/Neoadjuvant"]),
    ("Drug Class", "drug_class_grp", ["Immunotherapy (CPI-based)", "Small-Molecule Targeted Therapy", "Other"]),
    ("Trial Size", "trial_size", ["Small (<200)", "Medium (200–499)", "Large (≥500)"]),
    ("Start Era", "start_era", ["Pre-2015", "2015+"])
]

geo_labels = {
    "perc_pts_NorthAmerica_Europe": "Western enrollment (North America + Europe)",
    "_max_europe": "Europe or Western Europe",
    "_max_us": "US or North America"
}

analysis_df_summary = raw_df.copy()

# Western enrollment is a trial-level constant, and target-split rows duplicate the same
# comparison (identical OS HR and geography), so they are dropped from geography models.
# Arm-split rows are retained: same geography, but genuinely different OS HRs.
# Used only where the exposure is Western enrollment AND the outcome is OS
# (the S_GEO_FULL table and the geography-vs-OS rows of S_STRAT_ALL); the crossover and geography-vs-crossover
# tables keep every row.
df_geo_summary = build_dedup_geo_df(analysis_df_summary, TARGET_SPLIT_DROPS)
assert len(df_geo_summary) == len(analysis_df_summary) - len(TARGET_SPLIT_DROPS), (
    f"Geography dataset should drop {len(TARGET_SPLIT_DROPS)} rows, went from "
    f"{len(analysis_df_summary)} to {len(df_geo_summary)}"
)

# Crossover definitions whose denominator is the number of control patients who
# progressed.  A rate above 100% of that denominator is arithmetically impossible
# and marks an internally inconsistent report, so those comparisons are dropped
# wherever the definition is used -- as the exposure (crossover-vs-OS rows of
# S_STRAT_ALL) or as the outcome (geography-vs-crossover rows).  Its footnote says so.
#
PROGRESSED_DENOM_COLS = {col_exact_prog, col_inclass_prog}


def get_stats(sub_df, x_col, y_col):
    """Pearson correlation of x on y within a stratum.

    Returns the formatted strings the sheets show, plus the raw p-value (NaN when
    the correlation is not computable, i.e. fewer than 3 comparisons) and the
    number of distinct trials behind it, which the multiplicity step and the
    sheet footnotes read.
    """
    df_clean = sub_df[[x_col, y_col]].dropna()
    for _col in (x_col, y_col):
        if _col in PROGRESSED_DENOM_COLS:
            df_clean = df_clean[df_clean[_col] <= 1.0]
    n = len(df_clean)
    n_trials = int(sub_df.loc[df_clean.index, "study"].astype(str).str.strip().nunique())
    if n < 3:
        return {"n": n, "r_str": "-", "r2_str": "-", "p_str": "-", "p_raw": np.nan, "n_trials": n_trials}
    r, p = stats.pearsonr(df_clean[x_col], df_clean[y_col])
    r2 = r**2
    r_z = np.arctanh(r)
    with np.errstate(divide="ignore"):      # n = 3 gives an infinite standard error: the interval is the whole range
        se_z = 1 / np.sqrt(n - 3)
    z_low, z_high = r_z - 1.96 * se_z, r_z + 1.96 * se_z
    r_low, r_high = np.tanh(z_low), np.tanh(z_high)
    sign = "+" if r >= 0 else "−"
    sign_low = "+" if r_low >= 0 else "−"
    sign_high = "+" if r_high >= 0 else "−"
    r_str = f"{sign}{abs(r):.3f} ({sign_low}{abs(r_low):.3f} to {sign_high}{abs(r_high):.3f})"
    r2_str = f"{r2:.3f}"
    if p < 0.001: p_str = "<0.001*"
    elif p < 0.05: p_str = f"{p:.3f}*"
    else: p_str = f"{p:.3f}"
    return {"n": n, "r_str": r_str, "r2_str": r2_str, "p_str": p_str, "p_raw": float(p), "n_trials": n_trials}

all_cross_labels = {
    col_exact_sub: "Exact Drug (Denom: Subsequent Therapy)",
    col_inclass_sub: "Drug in Class (Denom: Subsequent Therapy)",
    col_exact_prog: "Exact Drug (Denom: Progressed)",
    col_inclass_prog: "Drug in Class (Denom: Progressed)",
    col_exact_event: "Exact Drug (Denom: PFS/EFS Event)",
    col_inclass_event: "Drug in Class (Denom: PFS/EFS Event)"
}

rows_all_defs = []
for x_col, x_label in all_cross_labels.items():
    for group_name, col_name, subgroups in STRATIFICATIONS_11:
        for subg in subgroups:
            sub_df = analysis_df_summary if col_name == "overall" else analysis_df_summary[analysis_df_summary[col_name] == subg]
            subg_label = "Overall (All trials)" if col_name == "overall" else subg
            st = get_stats(sub_df, x_col, "log_os_hr")
            flag_str = "Yes" if st["n"] < 10 else "No"
            rows_all_defs.append({
                "Analysis Type": "Crossover vs log OS HR",
                "Geographic Metric": "-",
                "Crossover Definition": x_label,
                "Stratification Group": group_name,
                "Subgroup": subg_label,
                "Small Subgroup Flag (<10 comparisons)": flag_str,
                "n": st["n"], "r (95% CI)": st["r_str"], "R²": st["r2_str"], "p-value": st["p_str"],
                "_p_raw": st["p_raw"], "_n_trials": st["n_trials"],
            })

for g_col, g_label in geo_labels.items():
    for x_col, x_label in all_cross_labels.items():
        for group_name, col_name, subgroups in STRATIFICATIONS_11:
            for subg in subgroups:
                sub_df = analysis_df_summary if col_name == "overall" else analysis_df_summary[analysis_df_summary[col_name] == subg]
                subg_label = "Overall (All trials)" if col_name == "overall" else subg
                st = get_stats(sub_df, g_col, x_col)
                flag_str = "Yes" if st["n"] < 10 else "No"
                rows_all_defs.append({
                    "Analysis Type": "Western enrollment vs crossover",
                    "Geographic Metric": g_label,
                    "Crossover Definition": x_label,
                    "Stratification Group": group_name,
                    "Subgroup": subg_label,
                    "Small Subgroup Flag (<10 comparisons)": flag_str,
                    "n": st["n"], "r (95% CI)": st["r_str"], "R²": st["r2_str"], "p-value": st["p_str"],
                    "_p_raw": st["p_raw"], "_n_trials": st["n_trials"],
                })

for g_col, g_label in geo_labels.items():
    for group_name, col_name, subgroups in STRATIFICATIONS_11:
        for subg in subgroups:
            # df_geo_summary, not analysis_df_summary: exposure is Western enrollment
            # and outcome is OS, so the target-split duplicates are dropped.
            sub_df = df_geo_summary if col_name == "overall" else df_geo_summary[df_geo_summary[col_name] == subg]
            subg_label = "Overall (All trials)" if col_name == "overall" else subg
            st = get_stats(sub_df, g_col, "log_os_hr")
            flag_str = "Yes" if st["n"] < 10 else "No"
            rows_all_defs.append({
                "Analysis Type": "Western enrollment vs log OS HR",
                "Geographic Metric": g_label,
                "Crossover Definition": "-",
                "Stratification Group": group_name,
                "Subgroup": subg_label,
                "Small Subgroup Flag (<10 comparisons)": flag_str,
                "n": st["n"], "r (95% CI)": st["r_str"], "R²": st["r2_str"], "p-value": st["p_str"],
                "_p_raw": st["p_raw"], "_n_trials": st["n_trials"],
            })

df_all_defs = pd.DataFrame(rows_all_defs)

# Assert total data rows in df_all_defs == 297
assert len(df_all_defs) == 297, f"Validation Error: df_all_defs has {len(df_all_defs)} rows, expected 297!"

# ── Multiplicity: Benjamini-Hochberg across every stratified correlation ─────
# The three stratified analyses are 297 correlations between them, of which the
# computable ones (3 or more comparisons in the stratum) are pooled and corrected
# together, since they are one family of exploratory looks at the same dataset.
# scipy.stats.false_discovery_control(method="bh") is the step-up procedure of
# Benjamini and Hochberg (1995); it returns the BH-adjusted p-values (q-values).
_computable = df_all_defs["_p_raw"].notna()
N_BH_TESTS = int(_computable.sum())
assert N_BH_TESTS == EXPECTED["n_stratified"], \
    f'Expected {EXPECTED["n_stratified"]} computable stratified correlations, got {N_BH_TESTS}'
assert (_computable == (df_all_defs["n"] >= 3)).all(), \
    "A stratum with 3 or more comparisons returned no p-value, or one with fewer returned one"
df_all_defs["_p_bh"] = np.nan
df_all_defs.loc[_computable, "_p_bh"] = stats.false_discovery_control(
    df_all_defs.loc[_computable, "_p_raw"].to_numpy(), method="bh")


def _fmt_adj_p(p_val):
    return "-" if pd.isna(p_val) else ("<0.001" if p_val < 0.001 else f"{p_val:.3f}")


BH_COL = f"BH-adjusted P ({N_BH_TESTS} tests)"
df_all_defs[BH_COL] = df_all_defs["_p_bh"].apply(_fmt_adj_p)

_nominal = df_all_defs["_p_raw"] < 0.05
assert (_nominal == df_all_defs["p-value"].str.endswith("*")).all(), \
    "The * marker and P < .05 disagree"
N_NOMINAL_ALL = int(_nominal.sum())
N_BH_SIG_ALL = int((df_all_defs["_p_bh"] < 0.05).sum())
SMALLEST_RAW_P = float(df_all_defs["_p_raw"].min())
print(f"Benjamini-Hochberg (scipy.stats.false_discovery_control, method='bh') across "
      f"{N_BH_TESTS} stratified correlations: smallest unadjusted P = {SMALLEST_RAW_P:.3f}; "
      f"{N_NOMINAL_ALL} with unadjusted P < .05; {N_BH_SIG_ALL} with BH-adjusted P < .05.")

# Split by association.  Each frame keeps its raw-p, BH and trial-count helper
# columns; the combined sheet below reads them for its sections and footnotes.
_df_xos_h = df_all_defs[df_all_defs["Analysis Type"] == "Crossover vs log OS HR"].drop(columns=["Geographic Metric"])
_df_gos_h = df_all_defs[df_all_defs["Analysis Type"] == "Western enrollment vs log OS HR"].drop(columns=["Crossover Definition"])
_df_gx_h  = df_all_defs[df_all_defs["Analysis Type"] == "Western enrollment vs crossover"].copy()

# The row counts are structural: 6 crossover definitions x 11 strata, 3 geographic
# metrics x 11 strata, and 3 x 6 x 11.  They must not move unless a stratification
# is added or removed.
assert len(_df_xos_h) == 66, f"Crossover-vs-OS row count = {len(_df_xos_h)}, expected 66"
assert len(_df_gos_h) == 33, f"Geography-vs-OS row count = {len(_df_gos_h)}, expected 33"
assert len(_df_gx_h) == 198, f"Geography-vs-crossover row count = {len(_df_gx_h)}, expected 198"

# Nominal-significance counts are result-dependent drift guards rather than structural ones: on this
# dataset 1, 2 and 4 correlations respectively reach P < .05 before correction.
assert int((_df_xos_h["_p_raw"] < 0.05).sum()) == 1, "Nominally significant crossover-vs-OS correlations: expected 1"
assert int((_df_gos_h["_p_raw"] < 0.05).sum()) == 2, "Nominally significant geography-vs-OS correlations: expected 2"
assert int((_df_gx_h["_p_raw"] < 0.05).sum()) == 4, "Nominally significant geography-vs-crossover correlations: expected 4"

print("PASSED row count assertions (xover-vs-OS=66, geo-vs-OS=33, geo-vs-xover=198, sum=297; "
      f"{N_BH_TESTS} computable).")

Benjamini-Hochberg (scipy.stats.false_discovery_control, method='bh') across 289 stratified correlations: smallest unadjusted P = 0.020; 7 with unadjusted P < .05; 0 with BH-adjusted P < .05.
PASSED row count assertions (xover-vs-OS=66, geo-vs-OS=33, geo-vs-xover=198, sum=297; 289 computable).


In [13]:
# ── S_STRAT_ALL: every computable stratified correlation on one sheet ────────
# The three association frames above are relabelled for display and stacked in
# analysis order (crossover vs OS, geography vs OS, geography vs crossover).  Nothing
# is recomputed: r, R² and P are the strings get_stats() produced, with the "*"
# marker dropped because the significant rows get their own section instead.
STRAT_ASSOC_ORDER = ["Crossover vs log OS HR", "Western enrollment vs log OS HR",
                     "Western enrollment vs crossover"]
STRAT_GEO_DISPLAY = {
    "Western enrollment (North America + Europe)": "Western (North America + Europe)",
    "Europe or Western Europe": "Europe or Western Europe",
    "US or North America": "US or North America",
    "-": "—",
}
STRAT_XOVER_DISPLAY = {
    "Exact Drug (Denom: Subsequent Therapy)": "Exact-drug / subsequent therapy",
    "Drug in Class (Denom: Subsequent Therapy)": "In-class / subsequent therapy",
    "Exact Drug (Denom: Progressed)": "Exact-drug / progressed",
    "Drug in Class (Denom: Progressed)": "In-class / progressed",
    "Exact Drug (Denom: PFS/EFS Event)": "Exact-drug / PFS/EFS event",
    "Drug in Class (Denom: PFS/EFS Event)": "In-class / PFS/EFS event",
    "-": "—",
}
STRAT_STRATUM_DISPLAY = {"Overall": "All trials", "Clinical Setting": "Clinical setting",
                         "Drug Class": "Drug class", "Trial Size": "Control-arm size",
                         "Start Era": "Enrollment start"}
STRAT_SUBGROUP_DISPLAY = {
    "Overall (All trials)": "All",
    "Immunotherapy (CPI-based)": "Checkpoint inhibitor",
    "Small-Molecule Targeted Therapy": "Small-molecule targeted therapy",
    "First-line (Metastatic/Advanced)": "First-line metastatic/advanced",
    "Adjuvant/Neoadjuvant": "Adjuvant/neoadjuvant",
    "2015+": "2015 or later",
}
assert set(STRAT_GEO_DISPLAY) - {"-"} == set(geo_labels.values())
assert set(STRAT_XOVER_DISPLAY) - {"-"} == set(all_cross_labels.values())
assert set(STRAT_STRATUM_DISPLAY) == {_grp for _grp, _, _ in STRATIFICATIONS_11}

STRAT_COLS = ["Association", "Geographic definition", "Crossover definition (numerator / denominator)",
              "Stratum", "Subgroup", "n", "r (95% CI)", "R²", "P"]
SMALL_N = 10

_strat_src = pd.concat([df_all_defs[df_all_defs["Analysis Type"] == _a] for _a in STRAT_ASSOC_ORDER])
assert len(_strat_src) == len(df_all_defs) == 297
_strat_noncomp = _strat_src[_strat_src["_p_raw"].isna()]
_strat = _strat_src[_strat_src["_p_raw"].notna()].reset_index(drop=True)
_strat_counts = {_a: int((_strat["Analysis Type"] == _a).sum()) for _a in STRAT_ASSOC_ORDER}
_strat_noncomp_counts = {_a: int((_strat_noncomp["Analysis Type"] == _a).sum()) for _a in STRAT_ASSOC_ORDER}
assert len(_strat) == N_BH_TESTS == EXPECTED["n_stratified"], \
    f'Expected {EXPECTED["n_stratified"]} computable correlations, got {len(_strat)}'
assert list(_strat_counts.values()) == EXPECTED["n_stratified_by_assoc"], f"Computable rows by association: {_strat_counts}"
assert len(_strat_noncomp) == 8, f"Expected 8 non-computable correlations, got {len(_strat_noncomp)}"
assert (_strat["r (95% CI)"] != "-").all() and (_strat_noncomp["r (95% CI)"] == "-").all()

df_strat_all = pd.DataFrame({
    "Association": _strat["Analysis Type"],
    "Geographic definition": _strat["Geographic Metric"].map(STRAT_GEO_DISPLAY),
    "Crossover definition (numerator / denominator)": _strat["Crossover Definition"].map(STRAT_XOVER_DISPLAY),
    "Stratum": _strat["Stratification Group"].map(STRAT_STRATUM_DISPLAY),
    "Subgroup": _strat["Subgroup"].map(lambda v: STRAT_SUBGROUP_DISPLAY.get(v, v)),
    "n": [f"{int(_n)}†" if _n < SMALL_N else int(_n) for _n in _strat["n"]],
    "r (95% CI)": _strat["r (95% CI)"],
    "R²": _strat["R²"],
    "P": _strat["p-value"].str.rstrip("*"),
}, columns=STRAT_COLS)
assert not df_strat_all.isna().any().any(), "A stratified-table label has no display mapping"

# Section 1: nominally significant rows, by displayed (3-decimal) P, ties in analysis order.
_sig_mask = (_strat["_p_raw"] < 0.05).to_numpy()
_p_display = df_strat_all["P"].map(lambda v: 0.0 if v == "<0.001" else float(v))
_sig_idx = sorted(np.flatnonzero(_sig_mask), key=lambda i: (_p_display.iloc[i], i))
_rest_idx = list(np.flatnonzero(~_sig_mask))
N_STRAT_SIG, N_STRAT_REST = len(_sig_idx), len(_rest_idx)
assert N_STRAT_SIG == N_NOMINAL_ALL == EXPECTED["n_nominal_sig"], \
    f'Expected {EXPECTED["n_nominal_sig"]} correlations with P < .05, got {N_STRAT_SIG}'
assert N_STRAT_SIG + N_STRAT_REST == len(df_strat_all)

# The footnote states that nothing survives correction and that every adjusted P exceeds .99.
_min_bh = float(_strat["_p_bh"].min())
assert N_BH_SIG_ALL == 0, f"{N_BH_SIG_ALL} correlations have BH-adjusted P < .05; footnote 4 would be wrong"
assert _min_bh > 0.99, f"Smallest BH-adjusted P is {_min_bh:.4f}, not above .99; footnote 4 would be wrong"

# ── Footnotes, every count read off the frames above ──
_n_xdefs, _n_gdefs = len(all_cross_labels), len(geo_labels)
N_STRATA = sum(len(_subs) for _, _, _subs in STRATIFICATIONS_11)
assert N_STRATA == 11, f"Expected 11 strata, got {N_STRATA}"


def _not_computable(assoc):
    _k = _strat_noncomp_counts[assoc]
    return f", {_k} not computable" if _k else ""


_strata_parts = ["all trials"] + [
    f"{STRAT_STRATUM_DISPLAY[_grp][0].lower()}{STRAT_STRATUM_DISPLAY[_grp][1:]} ({len(_subs)})"
    for _grp, _col, _subs in STRATIFICATIONS_11 if _col != "overall"]
fn_strat_1 = "Unweighted Pearson correlations by OLS (ordinary least squares) computed separately within each stratum."
fn_strat_2 = (
    f"{len(df_strat_all)} correlations: crossover vs log OS HR, {_strat_counts[STRAT_ASSOC_ORDER[0]]} "
    f"({_n_xdefs} crossover definitions × {N_STRATA} strata{_not_computable(STRAT_ASSOC_ORDER[0])}); "
    f"Western enrollment vs log OS HR, {_strat_counts[STRAT_ASSOC_ORDER[1]]} "
    f"({_n_gdefs} geographic definitions × {N_STRATA} strata{_not_computable(STRAT_ASSOC_ORDER[1])}); "
    f"Western enrollment vs crossover, {_strat_counts[STRAT_ASSOC_ORDER[2]]} "
    f"({_n_gdefs} × {_n_xdefs} × {N_STRATA}{_not_computable(STRAT_ASSOC_ORDER[2])}). "
    f"Strata: {'; '.join(_strata_parts)}."
)

# Footnote 3 describes the non-computable strata in words.  The sentence is assembled
# from the data, and its shape is asserted so a change in which strata fail stops the
# run instead of leaving a sentence that no longer describes the table.
_nc_geo = set(_strat_noncomp["Geographic Metric"])
assert _nc_geo == {"US or North America"}, f"Non-computable strata span {_nc_geo}"
_nc_by_sub = {_sub: _g for _sub, _g in _strat_noncomp.groupby("Subgroup")}
assert set(_nc_by_sub) == {"Large (≥500)", "Pre-2015"}, f"Non-computable subgroups: {sorted(_nc_by_sub)}"
_nc_large, _nc_pre = _nc_by_sub["Large (≥500)"], _nc_by_sub["Pre-2015"]
assert (_nc_large["Analysis Type"] == STRAT_ASSOC_ORDER[1]).sum() == 1
assert set(_nc_large.loc[_nc_large["Analysis Type"] == STRAT_ASSOC_ORDER[2], "Crossover Definition"]) \
    == set(all_cross_labels.values()) and len(_nc_large) == 1 + _n_xdefs
assert list(_nc_pre["Analysis Type"]) == [STRAT_ASSOC_ORDER[2]] \
    and list(_nc_pre["Crossover Definition"]) == [all_cross_labels[col_exact_prog]]
_pre_def = STRAT_XOVER_DISPLAY[all_cross_labels[col_exact_prog]].lower().replace(" / ", "/")
fn_strat_3 = (
    "Correlations were computed only for strata with at least 3 trials. "
    f"All {len(_strat_noncomp)} strata not computed involve {next(iter(_nc_geo))} enrollment: "
    f"control-arm size ≥500 (Western enrollment vs log OS HR, and Western enrollment vs crossover "
    f"under all {_n_xdefs} crossover definitions) and enrollment start before 2015 (Western "
    f"enrollment vs {_pre_def} crossover)."
)
fn_strat_4 = (
    f"P values are unadjusted. After Benjamini-Hochberg correction across all {N_BH_TESTS} "
    "correlations, no correlation remained significant (all adjusted P > .99). The "
    f"{N_STRAT_SIG} correlations with unadjusted P < .05 are listed first, in order of P; the "
    "remaining rows follow in analysis order."
)
fn_strat_5 = f"† Fewer than {SMALL_N} comparisons"
fn_strat_6 = ("Progressed-denominator crossover definitions exclude comparisons with crossover rates "
              f"above 100% ({TABLE_LABEL['S_IMPOSSIBLE']}).")
STRAT_FOOTNOTES = [fn_strat_1, fn_strat_2, fn_strat_3, fn_strat_4, fn_strat_5, fn_strat_6]

STRAT_CAPTION = (f"{TABLE_LABEL['S_STRAT_ALL']}. All {len(df_strat_all)} stratified correlations among "
                 "control-arm crossover, Western enrollment, and log OS HR.")
STRAT_SECTION_SIG = f"Nominally significant before correction (P < .05), n = {N_STRAT_SIG}"
STRAT_SECTION_REST = f"All other correlations, n = {N_STRAT_REST}"

# Layout: caption, blank, header, then two sections each opened by an italic label row
# in column A, then a blank row and the footnotes.  format_for_word() styles the sheet.
ws_strat = wb.create_sheet(title=SHEET_NAMES["S_STRAT_ALL"])
ws_strat.append([STRAT_CAPTION])
ws_strat.append([])
ws_strat.append(STRAT_COLS)
for _label, _idx in ((STRAT_SECTION_SIG, _sig_idx), (STRAT_SECTION_REST, _rest_idx)):
    ws_strat.append([_label])
    ws_strat.cell(row=ws_strat.max_row, column=1).font = Font(italic=True)
    for _i in _idx:
        ws_strat.append(list(df_strat_all.iloc[_i]))
ws_strat.append([])
for _fn in STRAT_FOOTNOTES:
    ws_strat.append([_fn])
ws_strat.freeze_panes = "A4"

print(f"\n{STRAT_CAPTION}")
print(f"  {STRAT_SECTION_SIG}; {STRAT_SECTION_REST}; smallest BH-adjusted P = {_min_bh:.3f}")
for _i, _fn in enumerate(STRAT_FOOTNOTES, start=1):
    print(f"  {_i}. {_fn}")


Supplementary Table S8. All 289 stratified correlations among control-arm crossover, Western enrollment, and log OS HR.
  Nominally significant before correction (P < .05), n = 7; All other correlations, n = 282; smallest BH-adjusted P = 0.999
  1. Unweighted Pearson correlations by OLS (ordinary least squares) computed separately within each stratum.
  2. 289 correlations: crossover vs log OS HR, 66 (6 crossover definitions × 11 strata); Western enrollment vs log OS HR, 32 (3 geographic definitions × 11 strata, 1 not computable); Western enrollment vs crossover, 191 (3 × 6 × 11, 7 not computable). Strata: all trials; clinical setting (2); drug class (3); control-arm size (3); enrollment start (2).
  3. Correlations were computed only for strata with at least 3 trials. All 8 strata not computed involve US or North America enrollment: control-arm size ≥500 (Western enrollment vs log OS HR, and Western enrollment vs crossover under all 6 crossover definitions) and enrollment start befor

## 8. Supplementary Tables S9-S10 - sensitivity analyses
S9 repeats the models with one comparison per trial; S10 caps progressed-denominator crossover rates above
100% at 100% instead of excluding them.

In [14]:
# Collapsed sensitivity table: the same models on one comparison per trial -- arm splits AND
# target splits collapsed, for all three exposures.
df_meta["_study_stripped"] = df_meta["study"].astype(str).str.strip()
df_collapsed = build_collapsed_df(df_meta, ARM_SPLIT_DROPS, TARGET_SPLIT_DROPS)
COLLAPSED_SPEC = [
    ("exact-drug crossover (primary)",              col_exact_sub,              100.0),
    ("in-class crossover",                          col_inclass_sub,            100.0),
    ("Western enrollment (North America + Europe)", "perc_pts_NorthAmerica_Europe",   1.0),
]
collapsed_rows = []
for _lab, _col, _scale in COLLAPSED_SPEC:
    for _adj in (False, True):
        _f = _fit_exposure(df_collapsed, _col, _adj, False, x_scale=_scale)
        collapsed_rows.append({
            "Exposure": _lab, "Model": "PFS-adjusted" if _adj else "Unadjusted",
            "Analyzed (k)": int(_f["k"]),
            "OS HR change (per +10pp)": _pct_of_ratio(_f["ratio"]),
            "95% CI (Hartung-Knapp)": f"{_pct_of_ratio(_f['lo'])} to {_pct_of_ratio(_f['hi'])}",
            "p-value": _fmt_p3(_f["p1"]), "tau²": f"{_f['tau2']:.5f}",
            "I² (%)": f"{_f['I2']:.1f}",
        })
df_collapsed_results = pd.DataFrame(collapsed_rows)

In [15]:
_write_table("S_COLLAPSED", f"{TABLE_LABEL['S_COLLAPSED']}. Sensitivity analysis of random-effects meta-regression for "
                    "crossover and Western enrollment vs log OS HR, collapsing each datapoint "
                    "to one comparison per trial.",
             df_collapsed_results, (PCT_SCALE_FOOTNOTE,))

In [16]:
# ── Crossover rates above 100% credited as 100% instead of excluded ──────────
# The full crossover and precision-bounds tables drop progressed-denominator rates above
# 100% as arithmetically impossible.  This sensitivity analysis keeps those comparisons
# and sets the rate to exactly 100%.  The "Excluded" half is the full crossover analysis
# re-fitted through the same machinery and checked against it below.
assert not (df_meta[col_exact_sub] > 1.0).any() and not (df_meta[col_inclass_sub] > 1.0).any() \
    and not (df_meta[col_exact_event] > 1.0).any() and not (df_meta[col_inclass_event] > 1.0).any(), \
    "A crossover column other than the progressed denominator now has values above 100%; " \
    "a new >100% column would need its own capped row in the capped-crossover table"

df_capped = df_meta.copy()
for _c in (col_exact_prog, col_inclass_prog):
    df_capped[_c] = df_capped[_c].clip(upper=1.0)
assert (df_meta[col_exact_prog] > 1.0).sum() == n_over_exact \
    and (df_meta[col_inclass_prog] > 1.0).sum() == n_over_inclass, \
    "df_meta must be left untouched by the capping"

CAPPED_DEFS = [("exact-drug", col_exact_prog), ("in-class", col_inclass_prog)]
CAPPED_STATS = ["k", "OS HR change (per +10pp)", "95% CI (Hartung-Knapp)", "p-value"]

capped_rows = []
for _num, _col in CAPPED_DEFS:
    for _adj in (False, True):
        _model = "PFS-adjusted" if _adj else "Unadjusted"
        _excl = _fit_exposure(df_meta,   _col, _adj, drop_impossible=True)
        _cap  = _fit_exposure(df_capped, _col, _adj, drop_impossible=False)
        _row = {"Numerator": _num, "Denominator": "progressed", "Model": _model}
        for _tag, _f in (("Excluded", _excl), ("Capped", _cap)):
            _row[f"{_tag}: k"] = int(_f["k"])
            _row[f"{_tag}: OS HR change (per +10pp)"] = _pct_of_ratio(_f["ratio"])
            _row[f"{_tag}: 95% CI (Hartung-Knapp)"] = f"{_pct_of_ratio(_f['lo'])} to {_pct_of_ratio(_f['hi'])}"
            _row[f"{_tag}: p-value"] = _fmt_p3(_f["p1"])
        # the rows the cap adds: original rate above 100% with every other field present
        _need = ["log_os_hr", "se_log_hr"] + (["log_pfs_hr"] if _adj else [])
        _expected_capped = int(((df_meta[_col] > 1.0) & df_meta[_need].notna().all(axis=1)).sum())
        assert _row["Capped: k"] - _row["Excluded: k"] == _expected_capped, \
            f"{TABLE_LABEL['S_CAPPED']} {_num} / {_model}: capped {_row['Capped: k'] - _row['Excluded: k']} " \
            f"!= expected {_expected_capped}"
        capped_rows.append(_row)

df_capped_results = pd.DataFrame(capped_rows)
assert len(df_capped_results) == 4, f"Expected 4 rows for {TABLE_LABEL['S_CAPPED']}, got {len(df_capped_results)}"

# The Excluded half must be the published analysis, not a re-implementation.
for _, _r in df_capped_results.iterrows():
    _t1 = df_xover_full[(df_xover_full["Numerator"] == _r["Numerator"]) & (df_xover_full["Denominator"] == "progressed")
                & (df_xover_full["Model"] == _r["Model"])]
    assert len(_t1) == 1, f"{TABLE_LABEL['S_CAPPED']} could not match exactly one {TABLE_LABEL['S_XOVER_FULL']} row"
    _pub = _t1.iloc[0]
    assert _r["Excluded: k"] == _pub["Analyzed (k)"] \
        and _r["Excluded: OS HR change (per +10pp)"] == _pub["OS HR change (per +10pp)"] \
        and _r["Excluded: 95% CI (Hartung-Knapp)"] == _pub["95% CI (Hartung-Knapp)"] \
        and _r["Excluded: p-value"] == _pub["p-value"], \
        f"{TABLE_LABEL['S_CAPPED']} excluded half disagrees with {TABLE_LABEL['S_XOVER_FULL']} ({_r['Numerator']}, {_r['Model']})"


def _capped_over_list(col):
    _o = df_meta[df_meta[col] > 1.0]
    return ", ".join(f"{s} ({r * 100:.1f}%)" for s, r in
                     sorted(zip(_o["study"].astype(str).str.strip(), _o[col]), key=lambda t: t[0].lower()))


CAPPED_FN1 = ("Comparisons whose progressed-denominator crossover rate exceeded 100% were set to 100% "
           f"instead of being excluded. Exact-drug: {_capped_over_list(col_exact_prog)}. "
           f"In-class: {_capped_over_list(col_inclass_prog)}.")
CAPPED_FN2 = ("Only the progressed denominator yields rates above 100%; the subsequent-therapy and "
           "PFS/EFS-event definitions are unaffected and are not repeated here.")
# Written with plain column titles; the Excluded / Capped band above them carries the group.
# A narrow blank column (H) between the two groups keeps their underlines apart.
_capped_out = df_capped_results[["Numerator", "Denominator", "Model"]
                  + [f"Excluded: {_s}" for _s in CAPPED_STATS]].copy()
_capped_out[""] = None
for _s in CAPPED_STATS:
    _capped_out[f"Capped: {_s}"] = df_capped_results[f"Capped: {_s}"]
_capped_out.columns = ["Numerator", "Denominator", "Model"] + CAPPED_STATS + [""] + CAPPED_STATS
_write_table("S_CAPPED", f"{TABLE_LABEL['S_CAPPED']}. Sensitivity analysis of Crossover vs log OS HR capping crossover "
                    "rates >100% instead of excluding them.",
             _capped_out, (CAPPED_FN1, CAPPED_FN2, PCT_SCALE_FOOTNOTE),
             group_headers=(("Excluded crossover rates >100%", 4, 7),
                            ("Capped crossover rates at 100%", 9, 12)))
print(f"Built {TABLE_LABEL['S_CAPPED']} ({len(df_capped_results)} rows): capping {n_over_exact} exact-drug and {n_over_inclass} "
      "in-class comparisons at 100% instead of excluding them.")
print(df_capped_results.to_string())

Built Supplementary Table S10 (4 rows): capping 2 exact-drug and 4 in-class comparisons at 100% instead of excluding them.
    Numerator Denominator         Model  Excluded: k Excluded: OS HR change (per +10pp) Excluded: 95% CI (Hartung-Knapp) Excluded: p-value  Capped: k Capped: OS HR change (per +10pp) Capped: 95% CI (Hartung-Knapp) Capped: p-value
0  exact-drug  progressed    Unadjusted           35                             +0.18%                 −1.42% to +1.80%             0.823         37                           −0.23%               −1.72% to +1.28%           0.753
1  exact-drug  progressed  PFS-adjusted           35                             +0.46%                 −1.01% to +1.95%             0.528         37                           +0.13%               −1.25% to +1.53%           0.849
2    in-class  progressed    Unadjusted           48                             +0.18%                 −1.75% to +2.15%             0.854         52                           −0.82%     

### Internal check - the comparisons excluded from the analyses
The comparisons dropped from the main analyses are recomputed independently and their composition is
asserted, so a change in the data that adds or removes an exclusion stops the run.

In [17]:
# ── Excluded comparisons: one record per (comparison, reason) ─────────────────
# One record per (comparison, reason); a comparison excluded for two reasons has two records.  There is no
# standard-error outlier: the widest SE of log OS HR is 0.46, far below the 1.5 guard.  IFM 2005-02 drops
# out because its OS hazard ratio has no confidence interval, so no standard error can be derived;
# IMvigor130 (monotherapy) has a usable standard error and is dropped only from PFS-adjusted models,
# which need a PFS hazard ratio.
EXCL_COLS = ["Study / Comparison", "Model Set Affected", "Reason for Exclusion"]
MULTI_ARM_REASON = "Multi-arm trial arm dropped to prevent double counting"
NO_CI_REASON = ("Overall survival hazard ratio reported without a confidence interval; "
                "no standard error of log OS HR could be derived")
NO_PFS_REASON = "PFS hazard ratio not reported"
GENERIC_ROW = {
    "Study / Comparison": "Progressed-denominator crossover > 100%",
    "Model Set Affected": "Progressed-denominator models",
    "Reason for Exclusion": "Arithmetically impossible crossover proportion (>100%)",
}

_multi_row_studies = set(raw_df["study"].astype(str).str.strip().value_counts().loc[lambda v: v > 1].index)

def _comparison_label(row):
    """Bare study name, or 'Study (regimen)' when the trial contributes >1 comparison."""
    _study = str(row["study"]).strip()
    return f"{_study} ({str(row['regimen']).strip()})" if _study in _multi_row_studies else _study

# 1. Impossible progressed-denominator crossover -- from the same source as the impossible-crossover table.
excl_impossible = [
    {"Study / Comparison": r["Study"],
     "Model Set Affected": "Progressed-denominator models",
     "Reason for Exclusion": r["Reason"]}
    for r in df_impossible.to_dict("records")
]

# 2. OS hazard ratio with no confidence interval -> no weight can be computed.
_no_ci = raw_df[raw_df["Combined_OS_HR"].notna() & raw_df["se_log_hr"].isna()]
excl_no_ci = [
    {"Study / Comparison": _comparison_label(r),
     "Model Set Affected": "All weighted analyses",
     "Reason for Exclusion": NO_CI_REASON}
    for _, r in _no_ci.iterrows()
]

# 3. Multi-arm / target-split arm drops -- the keep-lists defined in the configuration section.
#
# The two halves of that keep-list are dropped by different model sets.
#
# The four ARM-SPLIT rows are dropped only by the collapsed sensitivity analyses: the primary and
# principal-sensitivity meta-regressions keep both datapoints for those trials.
#
# The four TARGET-SPLIT rows are additionally dropped by every geography-vs-OS
# model.  Western enrollment is a trial-level constant and the two rows of a
# target-split trial report the same OS hazard ratio, so outside the crossover
# models they are the same observation entered twice.  They remain in the
# crossover models, where the two rows carry different crossover values.
MULTI_ARM_SCOPE = f"Collapsed sensitivity analyses only ({TABLE_LABEL['S_COLLAPSED']})"
TARGET_SPLIT_SCOPE = f"Geography-vs-OS models and collapsed sensitivity analyses ({TABLE_LABEL['S_COLLAPSED']})"
TARGET_SPLIT_REASON = (
    "Row duplicates the same comparison: the trial's two rows differ only in which drug of "
    "the experimental regimen carried a qualifying prior later-line approval, and report an "
    "identical OS hazard ratio and identical Western enrollment"
)
excl_multi_arm = (
    [{"Study / Comparison": f"{study} ({regimen})",
      "Model Set Affected": MULTI_ARM_SCOPE,
      "Reason for Exclusion": MULTI_ARM_REASON}
     for study, regimen in ARM_SPLIT_DROPS.items()]
    + [{"Study / Comparison": f"{study} ({specific} arm)",
        "Model Set Affected": TARGET_SPLIT_SCOPE,
        "Reason for Exclusion": TARGET_SPLIT_REASON}
       for study, specific in TARGET_SPLIT_DROPS.items()]
)

# Tie that scope to the numbers actually reported: the comparisons the drop list
# removes from the primary model must equal the gap between the primary k and the
# k printed on the collapsed-sensitivity sheet.
_arm_drop_mask = pd.Series(False, index=raw_df.index)
for _study, _regimen in ARM_SPLIT_DROPS.items():
    _arm_drop_mask |= ((raw_df["study"].astype(str).str.strip() == _study)
                       & (raw_df["regimen"].astype(str).str.strip() == _regimen))
for _study, _specific in TARGET_SPLIT_DROPS.items():
    _arm_drop_mask |= ((raw_df["study"].astype(str).str.strip() == _study)
                       & (raw_df["Specific Treatment"].astype(str).str.strip() == _specific))
assert int(_arm_drop_mask.sum()) == len(excl_multi_arm), (
    f"The arm-split drop list matches {int(_arm_drop_mask.sum())} rows but "
    f"{len(excl_multi_arm)} exclusion rows were built"
)

_primary_analyzed = (raw_df["log_os_hr"].notna() & raw_df[col_exact_sub].notna()
                     & raw_df["se_log_hr"].notna())
_k_primary = int(_primary_analyzed.sum())
_k_collapsed = int((_primary_analyzed & ~_arm_drop_mask).sum())

_ws_collapsed = wb[SHEET_NAMES["S_COLLAPSED"]]
_collapsed_header, _collapsed_rows = read_sheet_block(_ws_collapsed)
_collapsed_k = [r[_collapsed_header.index("Analyzed (k)")] for r in _collapsed_rows
          if str(r[0]).startswith("exact-drug crossover (primary)")
          and str(r[_collapsed_header.index("Model")]).strip() == "Unadjusted"]
assert len(_collapsed_k) == 1, f"Could not find the default unadjusted row on {SHEET_NAMES['S_COLLAPSED']}"
assert _collapsed_k[0] == _k_collapsed, (
    f"{SHEET_NAMES['S_COLLAPSED']} reports k={_collapsed_k[0]} for the collapsed unadjusted model, but applying "
    f"the arm-split drop list to the primary model gives k={_k_collapsed}"
)
print(f"Arm-split drops affect the collapsed sensitivity only: primary k={_k_primary}, "
      f"collapsed k={_k_collapsed} (matches {SHEET_NAMES['S_COLLAPSED']}).")

# 4. No PFS hazard ratio -> PFS-adjusted models only.
_no_pfs = raw_df[raw_df["Combined_OS_HR"].notna() & raw_df["log_pfs_hr"].isna()]
excl_no_pfs = [
    {"Study / Comparison": _comparison_label(r),
     "Model Set Affected": "PFS-adjusted models",
     "Reason for Exclusion": NO_PFS_REASON}
    for _, r in _no_pfs.iterrows()
]

excl_records = excl_impossible + excl_no_ci + excl_multi_arm + excl_no_pfs + [GENERIC_ROW]

df_excluded = pd.DataFrame(excl_records, columns=EXCL_COLS)
assert len(df_excluded) == len(
    df_excluded.drop_duplicates(subset=["Study / Comparison", "Reason for Exclusion"])
), "The excluded-comparison list contains duplicate (study, reason) pairs"

_excl_reason = df_excluded["Reason for Exclusion"].astype(str)
excl_composition = {
    "impossible crossover (numerator > progressed)": int(_excl_reason.str.contains("exceed control patients progressed|exceeds control patients progressed", regex=True).sum()),
    "no confidence interval on OS HR":               int((_excl_reason == NO_CI_REASON).sum()),
    "multi-arm trial arm dropped":                   int((_excl_reason == MULTI_ARM_REASON).sum()),
    "target-split duplicate comparison dropped":     int((_excl_reason == TARGET_SPLIT_REASON).sum()),
    "PFS hazard ratio not reported":                 int((_excl_reason == NO_PFS_REASON).sum()),
    "progressed-denominator crossover > 100%":       int(_excl_reason.str.contains("Arithmetically impossible crossover proportion").sum()),
}
assert excl_composition == {
    "impossible crossover (numerator > progressed)": EXPECTED["n_excluded_impossible"],
    "no confidence interval on OS HR":               1,
    "multi-arm trial arm dropped":                   4,
    "target-split duplicate comparison dropped":     4,
    "PFS hazard ratio not reported":                 1,
    "progressed-denominator crossover > 100%":       1,
}, f"Unexpected composition of the excluded comparisons: {excl_composition}"
assert len(df_excluded) == EXPECTED["n_excluded_total"], \
    f'Expected {EXPECTED["n_excluded_total"]} excluded-comparison rows, got {len(df_excluded)}'

# These counts are what the crossover and geography table footnotes state; keep them in step.
assert len(excl_no_ci) == 1 and excl_no_ci[0]["Study / Comparison"] == "IFM 2005-02", \
    f"The table footnotes name IFM 2005-02 as the only comparison without a CI; got {excl_no_ci}"
assert len(excl_no_pfs) == 1 and excl_no_pfs[0]["Study / Comparison"] == "IMvigor130 (Atezolizumab monotherapy)", \
    f"The table footnotes name IMvigor130 as the only comparison without a PFS HR; got {excl_no_pfs}"
assert set(df_excluded.loc[df_excluded["Reason for Exclusion"] == MULTI_ARM_REASON,
                          "Model Set Affected"]) == {MULTI_ARM_SCOPE}, \
    "Every arm-split drop row must be scoped to the collapsed sensitivity analyses"
assert set(df_excluded.loc[df_excluded["Reason for Exclusion"] == TARGET_SPLIT_REASON,
                          "Model Set Affected"]) == {TARGET_SPLIT_SCOPE}, \
    "Every target-split drop row must be scoped to the geography-vs-OS models as well"
assert (df_excluded["Study / Comparison"] == "IMvigor130 (Atezolizumab monotherapy)").sum() == 2, \
    "IMvigor130 (monotherapy) must appear twice: no PFS hazard ratio, and multi-arm arm drop"

Arm-split drops affect the collapsed sensitivity only: primary k=54, collapsed k=49 (matches S9. Collapsed Sensitivity).


## 9. Supplementary Tables S1-S3 - impossible crossover rates and the study list
S1 lists the comparisons whose progressed-denominator crossover rate exceeds 100%. S2 summarizes the
included trials and S3 lists them all.

In [18]:
_imp_caption = (f"{TABLE_LABEL['S_IMPOSSIBLE']}. Studies with crossover rates exceeding 100% "
                "(when calculated using the progressed denominator).")
ws_impossible = wb.create_sheet(title=SHEET_NAMES["S_IMPOSSIBLE"])
ws_impossible.append([_imp_caption])
ws_impossible.append([])
ws_impossible.append(list(df_impossible.columns))
for r in df_impossible.itertuples(index=False):
    ws_impossible.append(list(r))


def bold_drug(regimen, drug):
    """The regimen text with the qualifying drug in bold; a single-drug regimen stays plain."""
    at = regimen.lower().find(drug.lower())
    if at < 0 or regimen.lower() == f"{drug.lower()} monotherapy":
        return regimen
    parts = [regimen[:at], TextBlock(InlineFont(b=True), regimen[at:at + len(drug)]), regimen[at + len(drug):]]
    return CellRichText(*[p for p in parts if isinstance(p, TextBlock) or p != ""])


_regimen_col = IMPOSSIBLE_COL_ORDER.index("Regimen") + 1
for _i, (_regimen, _drug) in enumerate(zip(df_impossible["Regimen"], IMPOSSIBLE_PRIMARY_DRUG)):
    ws_impossible.cell(row=4 + _i, column=_regimen_col).value = bold_drug(_regimen, _drug)
ws_impossible.append([])
ws_impossible.append([f"{n_over_exact} comparisons exceed 100% under the exact-drug definition and "
              f"{n_over_inclass} under the in-class definition; every exact-drug violator also "
              f"violates in-class, so {len(df_impossible)} distinct comparisons are affected."])
ws_impossible.append(["Crossover Rate is the higher of the two progressed-denominator rates where both exceed 100%. "
                      "For regimens with multiple drugs, the treatment used for comparison is denoted in bold font."])

In [19]:
def create_nejm_table1(data_df):
    unique_df = collapse_to_one_row_per_trial(data_df)

    # A trial whose rows straddle two drug classes has no defensible study-level class unless a primary
    # agent is named for it.  Fail here rather than let spreadsheet row order pick one.
    _by_trial = (data_df.assign(_dc=data_df["Specific Treatment"].apply(classify_drug_class_from_specific))
                        .groupby(data_df["study"].astype(str).str.strip())["_dc"].nunique())
    _ambiguous = sorted(set(_by_trial[_by_trial > 1].index) - set(TARGET_SPLIT_DROPS))
    assert not _ambiguous, (
        f"These trials have rows in more than one drug class but no primary agent designated: {_ambiguous}. "
        "Add them to TARGET_SPLIT_DROPS rather than letting row order decide."
    )

    # Impute missing values and clean anomalies
    unique_df.loc[unique_df["study"] == "iNNOVATE", "condition_x"] = "Waldenstrom's macroglobulinemia"
    unique_df.loc[unique_df["study"] == "CALGB 100104", "ITT_Control_Num"] = 229

    # Define classification helper for Heme vs Solid
    heme_keywords = ["leukemia", "lymphoma", "myeloma", "leukaemia", "waldenstrom"]
    unique_df["is_heme"] = unique_df["condition_x"].fillna("").apply(
        lambda x: any(kw in x.lower() for kw in heme_keywords)
    )

    # Clean and parse enrollment start year
    def get_start_year(row):
        start_val = row.get("start")
        if pd.notna(start_val) and not isinstance(start_val, str):
            return start_val.year
        if pd.notna(start_val) and isinstance(start_val, str) and str(start_val).strip() != "":
            try:
                return pd.to_datetime(start_val).year
            except:
                pass
        enroll_val = row.get("enrollment")
        if pd.notna(enroll_val):
            match = re.search(r"(\d{4})", str(enroll_val))
            if match:
                return int(match.group(1))
        return np.nan

    unique_df["start_year"] = unique_df.apply(get_start_year, axis=1)

    # Classify solid and heme subcategories
    def solid_sub(row):
        if row["is_heme"]:
            return "Heme"
        cond = str(row["condition_x"]).lower()
        if "non-small cell" in cond or "nsclc" in cond:
            return "Non-small cell lung cancer"
        elif "small cell" in cond or "sclc" in cond:
            return "Small cell lung cancer"
        elif "lung" in cond:
            return "Non-small cell lung cancer"
        if "melanoma" in cond:
            return "Melanoma"
        if "prostate" in cond:
            return "Prostate cancer"
        if "breast" in cond:
            return "Breast cancer"
        if "renal cell" in cond or "rcc" in cond or "kidney" in cond:
            return "Clear cell renal cell carcinoma"
        if "colorectal" in cond or "colon" in cond:
            return "Colorectal cancer"
        if "urothelial" in cond or "bladder" in cond:
            return "Urothelial carcinoma"
        return "Other solid tumors"

    def heme_sub(row):
        if not row["is_heme"]:
            return "Solid"
        cond = str(row["condition_x"]).lower()
        if "multiple myeloma" in cond or "myeloma" in cond:
            return "Multiple myeloma"
        if "chronic lymphocytic" in cond or "cll" in cond:
            return "Chronic lymphocytic leukemia"
        return "Other hematologic malignancies"

    unique_df["solid_sub"] = unique_df.apply(solid_sub, axis=1)
    unique_df["heme_sub"] = unique_df.apply(heme_sub, axis=1)

    unique_df["drug_class"] = unique_df["Specific Treatment"].apply(classify_drug_class_from_specific)

    # Classify Context
    def group_context(c):
        c_str = str(c).lower()
        if "adjuvant" in c_str or "neoadjuvant" in c_str:
            return "Adjuvant/Neoadjuvant"
        else:
            return "First-line (Metastatic/Advanced)"

    unique_df["context_grouped"] = unique_df["context"].apply(group_context)

    # Classify Patient count buckets.
    # right=False makes the intervals left-closed -- [0,200), [200,500), [500,inf) -- so the printed labels
    # describe the bins exactly and match the `trial_size` strata of the stratified tables.  No trial in the
    # current set sits on a boundary.
    bins = [0, 200, 500, np.inf]
    labels = ["<200", "200–499", "≥500"]
    unique_df["pt_bucket"] = pd.cut(unique_df["ITT_Control_Num"], bins=bins,
                                    labels=labels, right=False)

    # Classify start year buckets
    year_bins = [1999, 2005, 2010, 2015, 2020, 2026]
    year_labels = ["2000-2005", "2006-2010", "2011-2015", "2016-2020", "2021-2025"]
    unique_df["year_bucket"] = pd.cut(unique_df["start_year"], bins=year_bins, labels=year_labels)

    N = len(unique_df)

    # Helper function to generate table lines
    def line(name, count, indent_level):
        indent = " " * indent_level
        pct = (count / N) * 100.0
        val_str = f"{count} ({pct:.1f}%)"
        return (f"{indent}{name}", val_str)

    # Generate rows structure
    table_rows = []
    
    # 1. Disease Type
    table_rows.append(("Disease Type", "section"))
    solid_count = (~unique_df["is_heme"]).sum()
    table_rows.append(line("Solid tumors", solid_count, 3))
    
    solid_subs = [
        ("Non-small cell lung cancer", "Non-small cell lung cancer"),
        ("Melanoma", "Melanoma"),
        ("Breast cancer", "Breast cancer"),
        ("Prostate cancer", "Prostate cancer"),
        ("Clear cell renal cell carcinoma", "Clear cell renal cell carcinoma"),
        ("Colorectal cancer", "Colorectal cancer"),
        ("Urothelial carcinoma", "Urothelial carcinoma"),
        ("Small cell lung cancer", "Small cell lung cancer"),
        ("Other solid tumors", "Other solid tumors"),
    ]
    for label, code in solid_subs:
        cnt = (unique_df["solid_sub"] == code).sum()
        if cnt > 0:
            table_rows.append(line(label, cnt, 6))

    heme_count = unique_df["is_heme"].sum()
    table_rows.append(line("Hematologic malignancies", heme_count, 3))
    
    heme_subs = [
        ("Multiple myeloma", "Multiple myeloma"),
        ("Chronic lymphocytic leukemia", "Chronic lymphocytic leukemia"),
        ("Other hematologic malignancies", "Other hematologic malignancies"),
    ]
    for label, code in heme_subs:
        cnt = (unique_df["heme_sub"] == code).sum()
        if cnt > 0:
            table_rows.append(line(label, cnt, 6))

    # 2. Drug Class
    table_rows.append(("Drug Class", "section"))
    drug_classes = [
        "Immunotherapy (CPI-based)",
        "Small-Molecule Targeted Therapy",
        "Monoclonal Antibody (non-CPI)",
        "Hormonal / Endocrine Therapy",
        "Immunomodulatory / Proteasome Inhibitor",
        "Chemotherapy / Cytotoxics",
        "Antibody-Drug Conjugate (ADC)",
        "Radioligand Therapy",
    ]
    for dc in drug_classes:
        cnt = (unique_df["drug_class"] == dc).sum()
        table_rows.append(line(dc, cnt, 3))

    # 3. Setting/Context
    table_rows.append(("Setting", "section"))
    table_rows.append(line("First-line (Metastatic/Advanced)", (unique_df["context_grouped"] == "First-line (Metastatic/Advanced)").sum(), 3))
    table_rows.append(line("Adjuvant/Neoadjuvant", (unique_df["context_grouped"] == "Adjuvant/Neoadjuvant").sum(), 3))

    # 4. ITT Control size
    table_rows.append(("Enrollment Size (Control ITT)", "section"))
    for _size_label in labels:
        table_rows.append(line(f"{_size_label} patients",
                               (unique_df["pt_bucket"] == _size_label).sum(), 3))

    # 5. Year of enrollment start
    table_rows.append(("Year of Enrollment Start", "section"))
    for yr in year_labels:
        cnt = (unique_df["year_bucket"] == yr).sum()
        table_rows.append(line(yr, cnt, 3))

    # 6. Sponsor Type
    table_rows.append(("Sponsor Type", "section"))
    sponsors = [
        ("Pharmaceutical industry", "Pharmaceutical industry"),
        ("Cooperative group", "Cooperative group"),
        ("Government", "Government"),
        ("Academic medical center", "Academic medical center"),
    ]
    for label, code in sponsors:
        cnt = (unique_df["sponsor_type"] == code).sum()
        table_rows.append(line(label, cnt, 3))

    # 7. Trials with Multiple Datapoints
    # 7. Trials Contributing Multiple Comparisons
    table_rows.append(("Trials Contributing Multiple Comparisons", "section"))
    study_counts = data_df['study'].value_counts()
    dup_studies = study_counts[study_counts > 1].index
    num_dup_studies = len(dup_studies)
    table_rows.append(line("Trials contributing multiple comparisons", num_dup_studies, 3))

    two_arm_count = 0
    for study in dup_studies:
        sub_df = data_df[data_df['study'] == study]
        unique_arms = sub_df.groupby(['regimen', 'comparator']).ngroups
        if unique_arms > 1:
            two_arm_count += 1

    two_crossover_count = 0
    for study in dup_studies:
        sub_df = data_df[data_df['study'] == study]
        spec_col = [col for col in sub_df.columns if col.lower() == 'specific treatment'][0]
        unique_treatments = sub_df[spec_col].dropna().unique()
        if len(unique_treatments) > 1:
            two_crossover_count += 1

    table_rows.append(line("Due to two experimental treatment arms", two_arm_count, 6))
    table_rows.append(line("Due to two crossover treatments evaluated", two_crossover_count, 6))

    # ── SUMMARY-OF-STUDIES SHEET ─────────────────────────────────────────────────────
    ws = wb.create_sheet(title=SHEET_NAMES["S_SUMMARY"])

    # Title row; fonts, widths, borders and merges are applied to every sheet by
    # format_for_word(), so only the semantic styling (bold section rows) is set here.
    ws["A1"] = f'{TABLE_LABEL["S_SUMMARY"]}. Summary of included studies.'

    # Column headers (Row 3)
    ws["A3"] = "Characteristic"
    ws["B3"] = f"Included Studies (N = {N})"

    # Data writing loop
    current_row = 4
    for item in table_rows:
        if item[1] == "section":
            ws.cell(row=current_row, column=1, value=item[0]).font = Font(bold=True)
            ws.cell(row=current_row, column=2, value="")
        else:
            ws.cell(row=current_row, column=1, value=item[0])
            ws.cell(row=current_row, column=2, value=item[1])
        current_row += 1

    last_data_row = current_row - 1

    # Footnote Row
    footnote_row = last_data_row + 2
    ws.cell(row=footnote_row, column=1, value=f"* A complete list of all {N} trials is provided in the Supplementary Appendix ({TABLE_LABEL['S_STUDIES']}).")

    # Freeze Panes
    ws.freeze_panes = "A4"

    # ── ALL-INCLUDED-STUDIES SHEET ─────────────────────────────────────────────────────
    ws2 = wb.create_sheet(title=SHEET_NAMES["S_STUDIES"])
    
    # Title Row (Row 1)
    ws2["A1"] = f"{TABLE_LABEL['S_STUDIES']}. All studies included in the analysis."

    # Headers (Row 3)
    headers_s2 = [
        ("A3", "Trial Name"),
        ("B3", "Enrollment Dates"),
        ("C3", "Experimental Therapy"),
        ("D3", "Control / Placebo"),
        ("E3", "ITT Control Size (N)"),
        ("F3", "Clinical Setting"),
        ("G3", "Drug Class"),
        ("H3", "Tumor Type"),
        ("I3", "Sponsor Type")
    ]
    for cell_id, title in headers_s2:
        ws2[cell_id] = title

    # Sort studies alphabetically
    sorted_df = unique_df.sort_values(by="study", key=lambda col: col.str.lower()).copy()

    # Write study details
    s2_row = 4
    for _, row in sorted_df.iterrows():
        ws2.cell(row=s2_row, column=1, value=row.get("study", ""))                # Trial Name
        ws2.cell(row=s2_row, column=2, value=row.get("enrollment", ""))           # Enrollment Dates
        ws2.cell(row=s2_row, column=3, value=row.get("regimen", ""))              # Experimental Therapy
        ws2.cell(row=s2_row, column=4, value=row.get("comparator", ""))           # Control / Placebo
        control_size = row.get("ITT_Control_Num", "")                             # ITT Control Size (N)
        ws2.cell(row=s2_row, column=5, value=int(control_size) if pd.notna(control_size) else "")
        ws2.cell(row=s2_row, column=6, value=row.get("context", ""))              # Clinical Setting
        ws2.cell(row=s2_row, column=7, value=row.get("drug_class", ""))           # Drug Class
        ws2.cell(row=s2_row, column=8, value=row.get("condition_x", ""))          # Tumor Type
        ws2.cell(row=s2_row, column=9, value=row.get("sponsor_type", ""))         # Sponsor Type
        s2_row += 1

    # Footnotes: a blank row, then the notes
    ws2.cell(row=s2_row + 1, column=1, value="Drug class reflects the trial's primary qualifying agent; trials contributing two comparisons may appear in different classes in Supplementary Tables S4–S10.")
    ws2.cell(row=s2_row + 2, column=1, value='Abbreviations: 177Lu-PSMA-617: Lutetium-177 vipivotide tetraxetan; A-AVD: Brentuximab vedotin, doxorubicin, vinblastine, and dacarbazine; ABCP: Atezolizumab, bevacizumab, carboplatin, and paclitaxel; ABVD: Doxorubicin, bleomycin, vinblastine, and dacarbazine; AC-TH: Doxorubicin and cyclophosphamide followed by a taxane with trastuzumab; ADC: Antibody-drug conjugate; ADT: Androgen deprivation therapy; ALK: Anaplastic lymphoma kinase; BR: Bendamustine and rituximab; BRAF: B-Raf proto-oncogene serine/threonine kinase; BTK: Bruton tyrosine kinase; BV-CHP: Brentuximab vedotin, cyclophosphamide, doxorubicin, and prednisone; CapeOx: Capecitabine and oxaliplatin; CE: Carboplatin and etoposide; CF: Cisplatin and fluorouracil; CHOP: Cyclophosphamide, doxorubicin, vincristine, and prednisone; CnP: Carboplatin and nab-paclitaxel; CP: Carboplatin and paclitaxel; CPI: Checkpoint inhibitor; CVb: Cisplatin and vinorelbine; Dara: Daratumumab; dd: Dose-dense; DC: Cisplatin and docetaxel; EC-TH: Epirubicin and cyclophosphamide followed by a taxane with trastuzumab; EGFR: Epidermal growth factor receptor; ET: Endocrine therapy; FAC-TH: Fluorouracil, doxorubicin, and cyclophosphamide followed by a taxane with trastuzumab; FCR: Fludarabine, cyclophosphamide, and rituximab; FEC-TH: Fluorouracil, epirubicin, and cyclophosphamide followed by a taxane with trastuzumab; FOLFIRI: Folinic acid, fluorouracil, and irinotecan; FOLFIRINOX: Folinic acid, fluorouracil, irinotecan, and oxaliplatin; FOLFOX: Folinic acid, fluorouracil, and oxaliplatin; GC: Cisplatin and gemcitabine; GCb: Carboplatin and gemcitabine; GClb: Obinutuzumab and chlorambucil; Isa: Isatuximab; ITT: Intention to treat; mFOLFOX6: Modified folinic acid, fluorouracil, and oxaliplatin regimen 6; mFOLFOX6-B: mFOLFOX6 with bevacizumab; MPT: Melphalan, prednisone, and thalidomide; nab-: Nanoparticle albumin-bound; NALIRIFOX: Liposomal irinotecan, fluorouracil, folinic acid, and oxaliplatin; PARP: Poly(ADP-ribose) polymerase; R-CHOP: Rituximab, cyclophosphamide, doxorubicin, vincristine, and prednisone; Rd: Lenalidomide and dexamethasone; RT: Radiotherapy; RVd: Lenalidomide, bortezomib, and dexamethasone; T-DM1: Ado-trastuzumab emtansine; T-DXd: Trastuzumab deruxtecan; TCH: Docetaxel, carboplatin, and trastuzumab; TH: Docetaxel and trastuzumab; THP: Docetaxel, trastuzumab, and pertuzumab; TKI: Tyrosine kinase inhibitor; TP: Cisplatin and paclitaxel; VMP: Bortezomib, melphalan, and prednisone; VR-CAP: Bortezomib, rituximab, cyclophosphamide, doxorubicin, and prednisone; VTD: Bortezomib, thalidomide, and dexamethasone.')

    # Freeze panes
    ws2.freeze_panes = "A4"

    # Save to file
    wb.save(RESULTS_WORKBOOK)
    print(f"Saved {TABLE_LABEL['S_SUMMARY']} and {TABLE_LABEL['S_STUDIES']} Excel sheets to: {RESULTS_WORKBOOK}")

create_nejm_table1(pd.read_excel(MASTER_FILE, sheet_name=SHEET))

Saved Supplementary Table S2 and Supplementary Table S3 Excel sheets to: outputs/Manuscript_Tables.xlsx


## 10. Workbook and Word export
The sheets are put in table order, formatted for pasting into Word (page-width columns, JAMA-style rules,
Arial), saved to `outputs/Manuscript_Tables.xlsx`, and exported to `outputs/Manuscript_Tables.docx`, which is
then checked against the workbook cell by cell.

In [20]:
excel_path = RESULTS_WORKBOOK

# ── 3. Sheet order + sheet-loss guardrail ────────────────────────────────────
# Reordering must never delete.  Anything unexpected in the workbook fails the
# `extra` assertion below; a sheet that genuinely has to go gets its own
# explicit `del wb[name]` with a comment saying why (see above).
final_sheet_order = [
    SHEET_NAMES["T_XOVER"], SHEET_NAMES["T_GEO"],
    SHEET_NAMES["S_IMPOSSIBLE"], SHEET_NAMES["S_SUMMARY"], SHEET_NAMES["S_STUDIES"],
    SHEET_NAMES["S_XOVER_FULL"], SHEET_NAMES["S_HETERO"], SHEET_NAMES["S_BOUNDS"],
    SHEET_NAMES["S_GEO_FULL"], SHEET_NAMES["S_STRAT_ALL"], SHEET_NAMES["S_COLLAPSED"],
    SHEET_NAMES["S_CAPPED"],
]
assert len(final_sheet_order) == len(set(final_sheet_order)) == 12, "final_sheet_order must list 12 distinct sheets"
# The tabs run in numeric order: T1, T2, then S1..S10.
assert [n.split(".")[0] for n in final_sheet_order[:12]] == ["T1", "T2"] + [f"S{i}" for i in range(1, 11)], \
    "Numbered tabs are out of order"
assert all(len(s) <= 31 for s in final_sheet_order), \
    f"Sheet names must be <= 31 chars: {[s for s in final_sheet_order if len(s) > 31]}"
assert all(":" not in s for s in final_sheet_order), \
    f"Sheet names must not contain ':': {[s for s in final_sheet_order if ':' in s]}"

missing = [s for s in final_sheet_order if s not in wb.sheetnames]
extra   = [s for s in wb.sheetnames if s not in final_sheet_order]
assert not missing, f"Expected sheets not found in workbook: {missing}"
assert not extra,   f"Workbook has sheets not in final_sheet_order: {extra}"

# ── Word-ready table formatting ──────────────────────────────────────────────
# Every sheet is sized to paste into a US Letter manuscript with 1-inch margins:
# usable width 624 px portrait / 864 px landscape at 96 dpi.  Column widths are
# computed from the cell contents, never typed in.
# Layout convention: row 1 caption, row 2 blank (or a group-header band), row 3
# column headers, row 4+ data until a blank row, then footnotes.
#
# Style (JAMA): Arial; caption 9 pt bold, header and body 8 pt, footnotes 7 pt; no
# vertical rules, one thin rule above and below the header and one below the last
# body row; statistic columns centred, text columns left-aligned.
# Only fonts, borders, alignment, widths, row heights, merges, page setup and
# freeze panes are touched -- no cell value is written here.  The one exception is
# _split_group_headers(), which is applied to Tables 1 and 2 and reports every header
# it rewrites.

WORD_FONT = "Arial"
PAGE_PX = {"portrait": 624, "landscape": 864}   # 6.5 in and 9.0 in at 96 dpi
PX_PER_CHAR_8PT = 5.5                           # estimated Arial advance, scaled by point size
CELL_PAD_PX = 10
MAX_NATURAL_CHARS = 40                          # cap on a column's natural width, in characters
SHRINK_STEP_PX = 4
CAPTION_PT, HEADER_PT, FOOTNOTE_PT = 9, 8, 7
_thin = Side(style="thin", color="000000")
MAIN_TABLES = (SHEET_NAMES["T_XOVER"], SHEET_NAMES["T_GEO"])


def _px(text, pt, bold=False):
    """Estimated width of `text`: 5.5 px per character at 8 pt, scaled by point size.  Capitals
    count as 1.25 characters and bold text is 8% wider, because 5.5 px is an average for
    lower-case text and would otherwise let "KEYNOTE-024" or a bold header word spill over."""
    chars = sum(1.25 if ch.isupper() else 1.0 for ch in text)
    return chars * PX_PER_CHAR_8PT * pt / 8 * (1.08 if bold else 1.0)


_PLACEHOLDERS = {"—", "–", "-", "NA", "N/A"}
_STAT_TOKEN = re.compile(r"^[<>≤≥+\-−]?\d*\.?\d+%?†?$")


def _stat_value(v):
    """True for a number or a statistic string ('0.020', '9†', '+0.18% (-1.38% to +1.77%)'),
    None for a blank or placeholder, False for text."""
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return True
    t = str(v).strip()
    if t == "" or t in _PLACEHOLDERS:
        return None
    toks = t.replace(" to ", " ").replace("(", " ").replace(")", " ").replace(",", " ").split()
    return all(_STAT_TOKEN.match(tok) for tok in toks)


# A line may break at whitespace and after a hyphen between letters ("Drug-class-adjusted");
# anything else, including a "|" between list items, is one unbreakable word.
_BREAKS = re.compile(r"\s+|(?<=[A-Za-z0-9]-)(?=[A-Za-z])")


def _words(text):
    return [w for w in _BREAKS.split(str(text)) if w]


def _table_layout(ws, header_row=3):
    n_cols = ws.max_column
    row = lambda r: [ws.cell(row=r, column=c).value for c in range(1, n_cols + 1)]
    last_data = header_row
    for r in range(header_row + 1, ws.max_row + 1):
        if _is_blank_row(row(r)):
            break
        last_data = r
    body_rows = list(range(header_row + 1, last_data + 1))
    return {
        "n_cols": n_cols, "header_row": header_row, "last_data": last_data, "body_rows": body_rows,
        "footnote_rows": [r for r in range(last_data + 1, ws.max_row + 1) if not _is_blank_row(row(r))],
        # a body row with text only in column A is a section label (S2, S8)
        "label_rows": [r for r in body_rows if not _is_blank_row(row(r)) and _is_blank_row(row(r)[1:])],
        "group_spans": [(m.min_col, m.max_col) for m in ws.merged_cells.ranges
                        if m.min_row == m.max_row == header_row - 1],
    }


def _column_specs(ws, lay, body_pt, header_pt=HEADER_PT):
    """Per column: statistic or text, and its natural and minimum width in px.
    Natural = longest line of any body cell (section-label rows excluded, capped at
    MAX_NATURAL_CHARS) or the header's longest word; minimum = longest single word anywhere."""
    specs = []
    for c in range(1, lay["n_cols"] + 1):
        hdr = ws.cell(row=lay["header_row"], column=c).value
        hdr_words = _words(hdr) if hdr is not None else []
        cells = [(r, ws.cell(row=r, column=c).value) for r in lay["body_rows"]]
        data = [v for r, v in cells if r not in lay["label_rows"] and v is not None]
        kinds = [_stat_value(v) for v in data]
        hdr_w = max((_px(w, header_pt, bold=True) for w in hdr_words), default=0)
        word_w = max((_px(w, body_pt) for _, v in cells if v is not None for w in _words(v)), default=0)
        line_w = max((min(_px(l, body_pt), _px("x" * MAX_NATURAL_CHARS, body_pt))
                      for v in data for l in str(v).split("\n")), default=0)
        min_px = max(word_w, hdr_w) + CELL_PAD_PX
        nat_px = max(line_w, hdr_w) + CELL_PAD_PX
        specs.append({"col": c,
                      "numeric": any(k is True for k in kinds) and all(k is not False for k in kinds),
                      "spacer": not hdr_words and not data,   # no title, no data: a gap between groups
                      "min": min_px, "nat": max(nat_px, min_px)})
    return specs


def _fit_widths(specs, limit):
    """Natural widths, with the widest text column shrunk toward its minimum until the table
    fits.  Statistic columns are never shrunk.  None when it cannot fit."""
    widths = [s["nat"] for s in specs]
    while sum(widths) > limit + 1e-9:
        over = sum(widths) - limit
        shrinkable = [i for i, s in enumerate(specs)
                      if not s["numeric"] and not s["spacer"] and widths[i] > s["min"]]
        if not shrinkable:
            return None
        i = max(shrinkable, key=lambda k: widths[k])
        widths[i] = max(specs[i]["min"], widths[i] - min(SHRINK_STEP_PX, over))
    return widths


def plan_word_layout(ws):
    """(orientation, body pt, header pt) for a sheet: the first option that fits, in the order the
    rules allow.  Tables 1 and 2 stay portrait at 8 pt.  A supplementary sheet drops to 7 pt body
    text only when it cannot fit landscape at 8 pt, and its header follows to 7 pt only when even
    that does not fit."""
    options = [("portrait", 8, 8)] if ws.title in MAIN_TABLES else \
        [("portrait", 8, 8), ("landscape", 8, 8), ("portrait", 7, 8), ("landscape", 7, 8), ("landscape", 7, 7)]
    lay = _table_layout(ws)
    for orientation, body_pt, header_pt in options:
        if _fit_widths(_column_specs(ws, lay, body_pt, header_pt), PAGE_PX[orientation]) is not None:
            return orientation, body_pt, header_pt
    return None


def _split_group_headers(ws, header_row=3):
    """'Unadjusted: P' -> 'Unadjusted' merged over the columns that share the prefix, with 'P'
    left as the column header.  Row 2 holds the band, so no row moves."""
    band_row, n_cols = header_row - 1, ws.max_column
    assert all(ws.cell(row=band_row, column=c).value in (None, "") for c in range(1, n_cols + 1)), \
        f"{ws.title}: row {band_row} is not empty, so it cannot hold a group-header band"
    head = {c: ws.cell(row=header_row, column=c).value for c in range(1, n_cols + 1)}
    c = 1
    while c <= n_cols:
        prefix = head[c].split(": ", 1)[0] if isinstance(head[c], str) and ": " in head[c] else None
        end = c
        while prefix and end < n_cols and isinstance(head[end + 1], str) and head[end + 1].startswith(prefix + ": "):
            end += 1
        if prefix and end > c:
            ws.cell(row=band_row, column=c).value = prefix
            for k in range(c, end + 1):
                new = head[k].split(": ", 1)[1]
                print(f"  {ws.title}!{get_column_letter(k)}{header_row}: {head[k]!r} -> {new!r}  "
                      f"(under {prefix!r} in {get_column_letter(k)}{band_row})")
                ws.cell(row=header_row, column=k).value = new
            ws.merge_cells(start_row=band_row, start_column=c, end_row=band_row, end_column=end)
        c = end + 1


def _wrapped_lines(text, width_px, pt, bold=False):
    """Estimated line count when `text` is word-wrapped into a cell `width_px` wide."""
    inner, space, lines = width_px - CELL_PAD_PX, _px(" ", pt, bold), 0
    for para in str(text).split("\n"):
        used, n = 0.0, 1
        for word in _words(para):
            w = _px(word, pt, bold)
            if used == 0:
                used = w
            elif used + space + w <= inner:
                used += space + w
            else:
                n, used = n + 1, w
        lines += n
    return lines


def format_for_word(ws, orientation, body_pt=8, header_pt=HEADER_PT, header_row=3):
    lay = _table_layout(ws, header_row)
    n_cols, last_data = lay["n_cols"], lay["last_data"]
    specs = _column_specs(ws, lay, body_pt, header_pt)
    widths = _fit_widths(specs, PAGE_PX[orientation])
    assert widths is not None, f"{ws.title} does not fit {orientation} at {body_pt}/{header_pt} pt"
    total_px = sum(widths)
    assert total_px <= PAGE_PX[orientation] + 1e-6, f"{ws.title}: {total_px:.1f} px exceeds {PAGE_PX[orientation]} px"
    numeric = {s["col"]: s["numeric"] for s in specs}
    group_spans = lay["group_spans"]
    band_row = header_row - 1

    # Styles go on before any merging, so every cell is an ordinary cell.
    for rng in list(ws.merged_cells.ranges):
        ws.unmerge_cells(str(rng))
    sides = defaultdict(dict)
    top_row = band_row if group_spans else header_row
    for c in range(1, n_cols + 1):
        sides[(top_row, c)]["top"] = _thin
        sides[(header_row, c)]["bottom"] = _thin
        sides[(last_data, c)]["bottom"] = _thin
    for lo, hi in group_spans:
        for c in range(lo, hi + 1):
            sides[(band_row, c)]["bottom"] = _thin   # spanner rule under each group

    for r in range(1, ws.max_row + 1):
        is_caption, is_header, is_body = r == 1, r == header_row, r in lay["body_rows"]
        is_footnote = r in lay["footnote_rows"]
        is_band = bool(group_spans) and r == band_row
        for c in range(1, n_cols + 1):
            cell = ws.cell(row=r, column=c)
            cell.border = Border(**sides[(r, c)])
            if is_caption:
                cell.font = Font(name=WORD_FONT, size=CAPTION_PT, bold=True)
                cell.alignment = Alignment(horizontal="left", vertical="top", wrap_text=True)
            elif is_band:
                cell.font = Font(name=WORD_FONT, size=header_pt, bold=True)
                cell.alignment = Alignment(horizontal="center", vertical="bottom")
            elif is_header:
                cell.font = Font(name=WORD_FONT, size=header_pt, bold=True)
                cell.alignment = Alignment(horizontal="center" if numeric[c] else "left",
                                           vertical="bottom", wrap_text=True)
            elif is_body:
                # keep any bold/italic that already carries meaning (S2 and S8 section rows)
                cell.font = Font(name=WORD_FONT, size=body_pt, bold=cell.font.b, italic=cell.font.i)
                if isinstance(cell.value, CellRichText):      # runs carry their own font
                    cell.value = CellRichText(*[
                        TextBlock(InlineFont(rFont=WORD_FONT, sz=body_pt, b=isinstance(p, TextBlock) and bool(p.font.b)),
                                  p if isinstance(p, str) else p.text) for p in cell.value])
                cell.alignment = Alignment(horizontal="center" if numeric[c] and r not in lay["label_rows"]
                                           else "left", vertical="top", wrap_text=True)
            elif is_footnote:
                cell.font = Font(name=WORD_FONT, size=FOOTNOTE_PT)
                cell.alignment = Alignment(horizontal="left", vertical="top", wrap_text=True)
            else:                                      # the blank rows between caption, header and footnotes
                cell.font = Font(name=WORD_FONT, size=body_pt)
        # header and body rows size themselves to their wrapped text; only the merged
        # caption and footnote rows, which Excel cannot auto-fit, get an explicit height
        ws.row_dimensions[r].height = None

    # Caption and footnotes: one merged band across the whole table, tall enough for all the text.
    if n_cols > 1:
        ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=n_cols)
        for r in lay["footnote_rows"]:
            ws.merge_cells(start_row=r, start_column=1, end_row=r, end_column=n_cols)
    for lo, hi in group_spans:
        ws.merge_cells(start_row=band_row, start_column=lo, end_row=band_row, end_column=hi)
    ws.row_dimensions[1].height = _wrapped_lines(ws["A1"].value, total_px, CAPTION_PT, bold=True) * 12
    for r in lay["footnote_rows"]:
        ws.row_dimensions[r].height = _wrapped_lines(ws.cell(row=r, column=1).value, total_px, FOOTNOTE_PT) * 10

    for spec, px in zip(specs, widths):
        ws.column_dimensions[get_column_letter(spec["col"])].width = (px - 5) / 7

    ws.freeze_panes = f"A{header_row + 1}"
    ws.print_title_rows = f"{band_row if group_spans else header_row}:{header_row}"
    ws.sheet_view.showGridLines = False
    ws.page_setup.orientation = orientation
    ws.page_setup.paperSize = ws.PAPERSIZE_LETTER
    ws.page_setup.fitToWidth, ws.page_setup.fitToHeight = 1, 0
    ws.sheet_properties.pageSetUpPr = PageSetupProperties(fitToPage=True)
    ws.page_margins = PageMargins(left=1, right=1, top=1, bottom=1, header=0.5, footer=0.5)

    return {"Sheet": ws.title, "Orientation": orientation, "Body pt": body_pt, "Header pt": header_pt, "Total px": round(total_px, 1),
            "Limit px": PAGE_PX[orientation], "Cols shrunk": sum(w < s["nat"] - 1e-9 for s, w in zip(specs, widths)),
            "Columns": n_cols}


WORD_LAYOUT = []
for _sheet_name in final_sheet_order:
    _ws = wb[_sheet_name]
    if _sheet_name in MAIN_TABLES:
        # Tables 1 and 2 share one two-level header: "Unadjusted" / "PFS-adjusted" bands over the
        # columns they cover, with the plain statistic names beneath.
        print(f"{_sheet_name}: moving the group prefixes of its headers into a band above them:")
        _split_group_headers(_ws)
    _plan = plan_word_layout(_ws)
    assert _plan is not None, f"{_sheet_name} does not fit a Letter page even at the smallest size allowed"
    assert _sheet_name not in MAIN_TABLES or _plan == ("portrait", 8, 8), \
        f"{_sheet_name} must be portrait at 8 pt, got {_plan}"
    WORD_LAYOUT.append(format_for_word(_ws, *_plan))
print(f"Formatted {len(final_sheet_order)} sheets for pasting into Word:")
print(pd.DataFrame(WORD_LAYOUT).to_string(index=False))

wb._sheets = [wb[s] for s in final_sheet_order]
wb.active = 0
# Pin every timestamp the .xlsx carries (document properties and zip member dates) so that re-running the
# notebook on unchanged inputs produces a byte-identical workbook.

wb.properties.created = datetime(2000, 1, 1)
wb.properties.modified = wb.properties.created

wb.save(excel_path)


def _freeze_xlsx_timestamps(path, when=(2000, 1, 1, 0, 0, 0)):
    """Rewrite an .xlsx with fixed zip member timestamps and a fixed core.xml date.

    openpyxl re-stamps `dcterms:modified` during save, so setting the property
    beforehand is not enough -- the value has to be rewritten in the saved file.
    """
    stamp = "%04d-%02d-%02dT%02d:%02d:%02dZ" % when
    tmp = f"{path}.tmp"
    with zipfile.ZipFile(path) as src, zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as dst:
        for item in sorted(src.infolist(), key=lambda i: i.filename):
            payload = src.read(item.filename)
            if item.filename == "docProps/core.xml":
                payload = re.sub(rb"(<dcterms:(?:created|modified)[^>]*>)[^<]*(</dcterms:)",
                                  rb"\g<1>" + stamp.encode() + rb"\g<2>", payload)
            info = zipfile.ZipInfo(item.filename, date_time=when)
            info.compress_type = zipfile.ZIP_DEFLATED
            info.external_attr = item.external_attr
            dst.writestr(info, payload)
    shutil.move(tmp, path)

T1. Crossover vs OS: moving the group prefixes of its headers into a band above them:
  T1. Crossover vs OS!D3: 'Unadjusted: OS HR change per +10 pp (95% CI)' -> 'OS HR change per +10 pp (95% CI)'  (under 'Unadjusted' in D2)
  T1. Crossover vs OS!E3: 'Unadjusted: P' -> 'P'  (under 'Unadjusted' in E2)
  T1. Crossover vs OS!F3: 'PFS-adjusted: OS HR change per +10 pp (95% CI)' -> 'OS HR change per +10 pp (95% CI)'  (under 'PFS-adjusted' in F2)
  T1. Crossover vs OS!G3: 'PFS-adjusted: P' -> 'P'  (under 'PFS-adjusted' in G2)
T2. Geo vs OS: moving the group prefixes of its headers into a band above them:
  T2. Geo vs OS!C3: 'Unadjusted: OS HR change per +10 pp (95% CI)' -> 'OS HR change per +10 pp (95% CI)'  (under 'Unadjusted' in C2)
  T2. Geo vs OS!D3: 'Unadjusted: P' -> 'P'  (under 'Unadjusted' in D2)
  T2. Geo vs OS!E3: 'PFS-adjusted: OS HR change per +10 pp (95% CI)' -> 'OS HR change per +10 pp (95% CI)'  (under 'PFS-adjusted' in E2)
  T2. Geo vs OS!F3: 'PFS-adjusted: P' -> 'P'  (under 

Formatted 12 sheets for pasting into Word:
                          Sheet Orientation  Body pt  Header pt  Total px  Limit px  Cols shrunk  Columns
            T1. Crossover vs OS    portrait        8          8     624.0       624            1        7
                  T2. Geo vs OS    portrait        8          8     624.0       624            1        6
       S1. Impossible Crossover    portrait        8          8     624.0       624            2        7
         S2. Summary of Studies    portrait        8          8     295.0       624            0        2
       S3. All Included Studies   landscape        7          8     864.0       864            8        9
   S4. Meta-reg Crossover vs OS   landscape        7          7     864.0       864            3       15
              S5. Heterogeneity    portrait        8          8     501.7       624            0        9
           S6. Precision Bounds   landscape        8          8     864.0       864            2        9
   

In [21]:
_freeze_xlsx_timestamps(excel_path)
print(f"Saved Excel workbook with {len(wb.sheetnames)} tabs to: {excel_path}")

Saved Excel workbook with 12 tabs to: outputs/Manuscript_Tables.xlsx


In [22]:
# ── WORD EXPORT: outputs/Manuscript_Tables.docx ──────────────────────────────
# Every Table and Supplementary Table (T1, T2, S1-S10; the REF tabs are skipped), in
# workbook order, one per section.  Everything -- cell text, column widths, orientation,
# font sizes, alignment and rules -- is read back from the saved workbook, so the document
# cannot drift from it.  Cell text is copied as stored, never reformatted.

DOCX_PATH = "outputs/Manuscript_Tables.docx"
WORD_SIDE_PX = 3            # Word cell margin each side; the other 4 of the workbook's 10 px padding is slack
_H_ALIGN = {"center": WD_ALIGN_PARAGRAPH.CENTER, "left": WD_ALIGN_PARAGRAPH.LEFT}


def _col_px(ws, n_cols):
    """Column widths in px, read from the saved sheet (a run of equal columns is one dimension)."""
    px = {}
    for dim in ws.column_dimensions.values():
        for c in range(dim.min, dim.max + 1):
            px[c] = dim.width * 7 + 5
    return [px[c] for c in range(1, n_cols + 1)]


def _px_to_in(px):
    return px / 96


def _set_run_font(run, pt, bold=False, italic=False):
    run.font.name = WORD_FONT
    run.font.size = Pt(pt)
    run.font.bold = bold
    run.font.italic = italic
    rfonts = run._r.get_or_add_rPr().rFonts
    for attr in ("w:eastAsia", "w:cs"):
        rfonts.set(qn(attr), WORD_FONT)


def _set_mark_size(par, pt):
    """Size of the paragraph mark, so an empty cell is not as tall as an 11 pt line."""
    pPr = par._p.get_or_add_pPr()
    rPr = OxmlElement("w:rPr")
    fonts = OxmlElement("w:rFonts")
    for attr in ("w:ascii", "w:hAnsi", "w:eastAsia", "w:cs"):
        fonts.set(qn(attr), WORD_FONT)
    sz = OxmlElement("w:sz")
    sz.set(qn("w:val"), str(round(pt * 2)))
    rPr.append(fonts)
    rPr.append(sz)
    pPr.insert_element_before(rPr, "w:sectPr", "w:pPrChange")


def _format_par(par, align=WD_ALIGN_PARAGRAPH.LEFT, before=0, after=0, keep_next=False):
    pf = par.paragraph_format
    pf.space_before, pf.space_after = Pt(before), Pt(after)
    pf.line_spacing_rule = WD_LINE_SPACING.SINGLE
    pf.keep_with_next = keep_next
    par.alignment = align


def _add_text_par(doc, text, pt, bold=False, align=WD_ALIGN_PARAGRAPH.LEFT, before=0, after=0, keep_next=False):
    par = doc.add_paragraph()
    _format_par(par, align, before, after, keep_next)
    _set_run_font(par.add_run(text), pt, bold=bold)
    _set_mark_size(par, pt)
    return par


def _set_cell_borders(cell, top=False, bottom=False):
    if not (top or bottom):
        return
    tcPr = cell._tc.get_or_add_tcPr()
    borders = OxmlElement("w:tcBorders")
    for edge, on in (("top", top), ("bottom", bottom)):
        if on:
            el = OxmlElement(f"w:{edge}")
            el.set(qn("w:val"), "single")
            el.set(qn("w:sz"), "4")        # 1/8 pt units: a half-point rule
            el.set(qn("w:space"), "0")
            el.set(qn("w:color"), "000000")
            borders.append(el)
    tcPr.insert_element_before(borders, "w:shd", "w:noWrap", "w:tcMar", "w:textDirection",
                               "w:tcFitText", "w:vAlign", "w:hideMark")


def _write_cell(cell, text, pt, bold, italic, align, valign, top=False, bottom=False, runs=None):
    par = cell.paragraphs[0]
    _format_par(par, _H_ALIGN[align])
    if runs:                                   # rich text: (text, bold) pieces
        for run_text, run_bold in runs:
            _set_run_font(par.add_run(run_text), pt, bold=bold or run_bold, italic=italic)
    elif text != "":
        _set_run_font(par.add_run(text), pt, bold=bold, italic=italic)
    _set_mark_size(par, pt)
    _set_cell_borders(cell, top=top, bottom=bottom)
    cell.vertical_alignment = valign


def _add_word_table(doc, ws, lay, widths_px):
    n_cols, header_row = lay["n_cols"], lay["header_row"]
    spans = {lo: hi for lo, hi in lay["group_spans"]}
    first_row = header_row - 1 if spans else header_row
    rows = list(range(first_row, lay["last_data"] + 1))
    table = doc.add_table(rows=len(rows), cols=n_cols)
    table.autofit = False
    tblPr = table._tbl.tblPr
    tblPr.find(qn("w:tblW")).set(qn("w:type"), "dxa")
    tblPr.find(qn("w:tblW")).set(qn("w:w"), str(round(sum(widths_px) / 96 * 1440)))
    margins = OxmlElement("w:tblCellMar")
    for edge, dxa in (("top", 15), ("left", WORD_SIDE_PX * 15), ("bottom", 15), ("right", WORD_SIDE_PX * 15)):
        el = OxmlElement(f"w:{edge}")
        el.set(qn("w:w"), str(dxa))
        el.set(qn("w:type"), "dxa")
        margins.append(el)
    tblPr.find(qn("w:tblLook")).addprevious(margins)
    for c, px in enumerate(widths_px):
        table.columns[c].width = Inches(_px_to_in(px))

    for t_idx, r in enumerate(rows):
        row = table.rows[t_idx]
        trPr = row._tr.get_or_add_trPr()
        trPr.append(OxmlElement("w:cantSplit"))
        if r <= header_row:                               # the band (if any) and the header repeat on every page
            trPr.append(OxmlElement("w:tblHeader"))
        # a group-header band merges across its columns; every other cell is its own
        cells, c = [], 1
        while c <= n_cols:
            hi = spans[c] if (r == header_row - 1 and c in spans) else c
            cell = row.cells[c - 1].merge(row.cells[hi - 1]) if hi > c else row.cells[c - 1]
            cell.width = Inches(_px_to_in(sum(widths_px[c - 1:hi])))
            cells.append((c, cell))
            c = hi + 1
        for c, cell in cells:
            src = ws.cell(row=r, column=c)
            v = src.value
            _write_cell(cell, "" if v is None else str(v), src.font.sz, bool(src.font.b), bool(src.font.i),
                        src.alignment.horizontal or "left",
                        WD_ALIGN_VERTICAL.BOTTOM if r <= header_row else WD_ALIGN_VERTICAL.TOP,
                        top=src.border.top is not None and src.border.top.style is not None,
                        bottom=src.border.bottom is not None and src.border.bottom.style is not None,
                        runs=[(p if isinstance(p, str) else p.text, not isinstance(p, str) and bool(p.font.b)) for p in v]
                        if isinstance(v, CellRichText) else None)
    return table


def build_manuscript_docx(path):
    wb_word = openpyxl.load_workbook(excel_path, rich_text=True)
    doc = Document()
    for el in doc.settings.element.find(qn("w:compat")).findall(qn("w:compatSetting")):
        if el.get(qn("w:name")) == "compatibilityMode":
            el.set(qn("w:val"), "15")       # Word 2013+ layout: the table edge sits on the margin
    normal = doc.styles["Normal"]
    normal.font.name, normal.font.size = WORD_FONT, Pt(8)
    normal.paragraph_format.space_after = Pt(0)
    normal.paragraph_format.line_spacing = 1.0
    doc.core_properties.title = "Manuscript tables"
    doc.core_properties.author = ""
    doc.core_properties.created = doc.core_properties.modified = datetime(2000, 1, 1)

    names = [n for n in final_sheet_order if not n.startswith("REF")]
    for i, name in enumerate(names):
        ws = wb_word[name]
        lay = _table_layout(ws)
        widths_px = _col_px(ws, lay["n_cols"])
        if i == 0:
            section = doc.sections[0]
        else:
            section = doc.add_section(WD_SECTION.NEW_PAGE)
            # the paragraph that carries the previous section's break: one point tall, so a
            # table that fills its page does not spill a blank line onto a new one
            brk = Paragraph(doc.element.body.findall(qn("w:p"))[-1], doc._body)
            _format_par(brk)
            brk.paragraph_format.line_spacing_rule = WD_LINE_SPACING.EXACTLY
            brk.paragraph_format.line_spacing = Pt(1)
            _set_mark_size(brk, 1)
        landscape = ws.page_setup.orientation == "landscape"
        section.orientation = WD_ORIENT.LANDSCAPE if landscape else WD_ORIENT.PORTRAIT
        section.page_width, section.page_height = (Inches(11), Inches(8.5)) if landscape else (Inches(8.5), Inches(11))
        section.left_margin = section.right_margin = section.top_margin = section.bottom_margin = Inches(1)
        section.header_distance = section.footer_distance = Inches(0.5)

        _add_text_par(doc, str(ws["A1"].value), ws["A1"].font.sz, bold=True, after=4, keep_next=True)
        _add_word_table(doc, ws, lay, widths_px)
        for k, r in enumerate(lay["footnote_rows"]):
            _add_text_par(doc, str(ws.cell(row=r, column=1).value), ws.cell(row=r, column=1).font.sz,
                          before=4 if k == 0 else 0, after=2)
    doc.save(path)
    _freeze_xlsx_timestamps(path)      # same fixed zip dates as the workbook, so reruns are byte-identical
    return names


word_names = build_manuscript_docx(DOCX_PATH)
print(f"Saved {DOCX_PATH}: {len(word_names)} tables in workbook order ({word_names[0]} ... {word_names[-1]}).")


# ── Check: the document says exactly what the workbook says ──────────────────
def _body_sequence(doc):
    """For each table: (caption text, [footnote texts]) read off the document body."""
    out = []
    for tbl in doc.element.body.findall(qn("w:tbl")):
        cap = tbl.getprevious()
        assert cap is not None and cap.tag == qn("w:p"), "a table has no caption paragraph above it"
        notes, nxt = [], tbl.getnext()
        while nxt is not None and nxt.tag == qn("w:p") and nxt.find(qn("w:pPr") + "/" + qn("w:sectPr")) is None:
            notes.append("".join(t.text or "" for t in nxt.iter(qn("w:t"))))
            nxt = nxt.getnext()
        out.append(("".join(t.text or "" for t in cap.iter(qn("w:t"))), notes))
    return out


def verify_manuscript_docx(path):
    wb_word = openpyxl.load_workbook(excel_path, rich_text=True)
    doc = Document(path)
    names = [n for n in final_sheet_order if not n.startswith("REF")]
    assert len(doc.tables) == len(doc.sections) == len(names), \
        f"{len(doc.tables)} tables / {len(doc.sections)} sections for {len(names)} sheets"
    seq = _body_sequence(doc)
    problems = []
    for name, table, section, (caption, notes) in zip(names, doc.tables, doc.sections, seq):
        ws = wb_word[name]
        lay = _table_layout(ws)
        spans = {lo: hi for lo, hi in lay["group_spans"]}
        first_row = lay["header_row"] - 1 if spans else lay["header_row"]
        sheet_rows = list(range(first_row, lay["last_data"] + 1))
        if caption != str(ws["A1"].value):
            problems.append((name, "caption", caption[:60]))
        if notes != [str(ws.cell(row=r, column=1).value) for r in lay["footnote_rows"]]:
            problems.append((name, "footnotes", f"{len(notes)} paragraphs vs {len(lay['footnote_rows'])} rows"))
        if len(table.rows) != len(sheet_rows):
            problems.append((name, "row count", f"{len(table.rows)} vs {len(sheet_rows)}"))
            continue
        for t_idx, r in enumerate(sheet_rows):
            seen, got = None, []
            for cell in table.rows[t_idx].cells:        # a merged cell repeats once per grid column
                got.append("" if cell._tc is seen else cell.text)
                seen = cell._tc
            want = ["" if ws.cell(row=r, column=c).value is None else str(ws.cell(row=r, column=c).value)
                    for c in range(1, lay["n_cols"] + 1)]
            if got != want:
                problems.append((name, f"row {r}", [(a, b) for a, b in zip(got, want) if a != b][:2]))
            for c in range(1, lay["n_cols"] + 1):          # bold runs (the qualifying drug in S1)
                v = ws.cell(row=r, column=c).value
                if isinstance(v, CellRichText):
                    want_bold = [p.text for p in v if not isinstance(p, str) and p.font.b]
                    got_bold = [run.text for par in table.rows[t_idx].cells[c - 1].paragraphs for run in par.runs if run.bold]
                    if got_bold != want_bold:
                        problems.append((name, f"row {r}", f"bold text {got_bold} vs {want_bold}"))
            trPr = table.rows[t_idx]._tr.trPr
            is_hdr = trPr is not None and trPr.find(qn("w:tblHeader")) is not None
            if is_hdr != (r <= lay["header_row"]):
                problems.append((name, f"row {r}", "header-row repeat flag is wrong"))
            if trPr is None or trPr.find(qn("w:cantSplit")) is None:
                problems.append((name, f"row {r}", "row may split across pages"))
        want_orient = WD_ORIENT.LANDSCAPE if ws.page_setup.orientation == "landscape" else WD_ORIENT.PORTRAIT
        if section.orientation != want_orient:
            problems.append((name, "orientation", str(section.orientation)))
        usable = section.page_width - section.left_margin - section.right_margin
        grid = sum(int(g.get(qn("w:w"))) for g in table._tbl.tblGrid.findall(qn("w:gridCol")))
        if grid > usable / 635:                          # twips; 1 twip = 635 EMU
            problems.append((name, "width", f"table {grid} twips wider than {usable / 635:.0f}"))
        tblLayout = table._tbl.tblPr.find(qn("w:tblLayout"))
        if tblLayout is None or tblLayout.get(qn("w:type")) != "fixed":
            problems.append((name, "layout", "autofit is on"))
    return problems


_docx_problems = verify_manuscript_docx(DOCX_PATH)
for _p in _docx_problems:
    print("  MISMATCH:", _p)
assert not _docx_problems, f"{len(_docx_problems)} differences between {DOCX_PATH} and the workbook"
print(f"{DOCX_PATH} matches the workbook: captions, cell text row by row, footnotes, repeating header rows, "
      f"section orientation and column widths for all {len(word_names)} tables.")

Saved outputs/Manuscript_Tables.docx: 12 tables in workbook order (T1. Crossover vs OS ... S10. Crossover Capped at 100%).


outputs/Manuscript_Tables.docx matches the workbook: captions, cell text row by row, footnotes, repeating header rows, section orientation and column widths for all 12 tables.


## 11. Figure 2 - meta-regression of OS hazard ratio on crossover and on Western enrollment

In [23]:
# ── PRIMARY PUBLICATION FIGURE 2 (META-REGRESSION) ───────────────────────────

df_fig2 = pd.read_excel(MASTER_FILE, sheet_name=SHEET)

num_cols = [
    "Combined_OS_HR", "Combined_OS_HR_CI_low", "Combined_OS_HR_CI_high",
    "Frac_ctrl_pts_getting_exact_drug_post-protocol_progressed_subsequent_therapy",
    "perc_pts_NorthAmerica_Europe"
]
for c in num_cols:
    df_fig2[c] = pd.to_numeric(df_fig2[c], errors="coerce")

def parse_geo_pct(val):
    if pd.isna(val): return np.nan
    if isinstance(val, (int, float)):
        if val == 0: return 0.0
        elif val <= 1.0: return val * 100.0
        else: return float(val)
    val_str = str(val).strip().lower()
    if val_str in ["na", "no data", "nr", "nan", ""]: return np.nan
    match = re.search(r"([0-9.]+)\s*%", val_str)
    if match:
        return float(match.group(1))
    match_num = re.search(r"([0-9.]+)", val_str)
    if match_num:
        num = float(match_num.group(1))
        if num <= 1.0: return num * 100.0
        else: return num
    return np.nan

df_fig2["perc_pts_NorthAmerica_Europe"] = df_fig2["perc_pts_NorthAmerica_Europe"].apply(parse_geo_pct)

df_fig2["log_hr"] = np.log(df_fig2["Combined_OS_HR"])
df_fig2["se_log_hr"] = (np.log(df_fig2["Combined_OS_HR_CI_high"]) - np.log(df_fig2["Combined_OS_HR_CI_low"])) / (2 * 1.959964)
df_fig2["v"] = df_fig2["se_log_hr"]**2

col_exact = "Frac_ctrl_pts_getting_exact_drug_post-protocol_progressed_subsequent_therapy"
col_geo = "perc_pts_NorthAmerica_Europe"

def meta_reg(y, v, X):
    k, p = X.shape
    tau2 = 0.0
    for it in range(200):
        w = 1.0 / (v + tau2)
        W = np.diag(w)
        XtW = X.T @ W
        XtWX_inv = np.linalg.inv(XtW @ X)
        P = W - W @ X @ XtWX_inv @ XtW
        PP = P @ P
        score = y.T @ PP @ y - np.trace(P)
        info = np.trace(PP)
        step = score / info
        tau2_new = max(0.0, tau2 + step)
        if abs(tau2_new - tau2) < 1e-10:
            tau2 = tau2_new
            break
        tau2 = tau2_new
    
    w = 1.0 / (v + tau2)
    W = np.diag(w)
    XtW = X.T @ W
    XtWX_inv = np.linalg.inv(XtW @ X)
    beta = XtWX_inv @ XtW @ y
    r = y - X @ beta
    q = np.sum(w * r**2) / (k - p)
    var_hk = q * XtWX_inv
    se_hk = np.sqrt(np.diag(var_hk))
    t_stat = beta / se_hk
    p_hk = 2 * stats.t.sf(np.abs(t_stat), df=k-p)
    return {"k": k, "beta": beta, "se_hk": se_hk, "p_hk": p_hk, "tau2": tau2, "q": q, "XtWX_inv": XtWX_inv, "v": v}

def fmt_p_jama(p_val):
    """JAMA style P value: italic P, no leading zero, '<.001' below the reporting floor."""
    if p_val < 0.001:
        return "$P$ < .001"
    return "$P$ = " + f"{p_val:.3f}".lstrip("0")


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
fig.subplots_adjust(wspace=0.30, left=0.08, right=0.96, top=0.88, bottom=0.15)

# Panel 2A: Exact-Drug Crossover vs log OS HR
m1 = df_fig2[df_fig2["Combined_OS_HR"].notna() & df_fig2["Combined_OS_HR_CI_low"].notna() & df_fig2["se_log_hr"].notna() & (df_fig2["se_log_hr"] > 0) & df_fig2[col_exact].notna()].copy()
y1 = m1["log_hr"].values
v1 = m1["v"].values
x1_pct = m1[col_exact].values * 100.0
x1_scaled = x1_pct / 10.0
X1 = np.column_stack([np.ones(len(y1)), x1_scaled])
res1 = meta_reg(y1, v1, X1)

w1 = 1.0 / (v1 + res1["tau2"])
weights_norm1 = 20.0 + (w1 - w1.min()) / (w1.max() - w1.min() + 1e-12) * 180.0

ax1.scatter(x1_pct, y1, s=weights_norm1, color="#1f77b4", alpha=0.6, edgecolors="black", linewidth=0.5, zorder=3)

x_grid1_pct = np.linspace(x1_pct.min(), x1_pct.max(), 200)
x_grid1_scaled = x_grid1_pct / 10.0
X_grid1 = np.column_stack([np.ones(len(x_grid1_pct)), x_grid1_scaled])
y_fit1 = X_grid1 @ res1["beta"]

se_fit1 = np.sqrt(np.sum((X_grid1 @ (res1["q"] * res1["XtWX_inv"])) * X_grid1, axis=1))
t_crit1 = stats.t.ppf(0.975, df=res1["k"]-2)
ci_low1 = y_fit1 - t_crit1 * se_fit1
ci_high1 = y_fit1 + t_crit1 * se_fit1

ax1.plot(x_grid1_pct, y_fit1, color="grey", linestyle="--", linewidth=2, zorder=4)
ax1.fill_between(x_grid1_pct, ci_low1, ci_high1, color="#1f77b4", alpha=0.15, zorder=2)

yticks = np.log([0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.25])
yticklabels = ["0.4", "0.5", "0.6", "0.7", "0.8", "0.9", "1.0", "1.25"]
ax1.set_yticks(yticks)
ax1.set_yticklabels(yticklabels)
ax1.set_xlabel("% Control Group Exact-Drug Crossover", fontsize=11, fontweight="bold")
ax1.set_ylabel("Overall Survival Hazard Ratio (log scale)", fontsize=11, fontweight="bold")
ax1.set_title("A  Exact-Drug Crossover vs Overall Survival", fontsize=12, fontweight="bold", loc="left", pad=10)

p1 = res1["p_hk"][1]

stats_text1 = f"Number of comparisons = {res1['k']}\n{fmt_p_jama(p1)}"
ax1.text(0.04, 0.94, stats_text1, transform=ax1.transAxes, fontsize=9.0, va="top", ha="left",
         bbox=dict(boxstyle="round,pad=0.5", facecolor="white", edgecolor="#cccccc", alpha=0.95), zorder=5)

# Panel 2B: North America + Europe Enrollment vs log OS HR
# Western enrollment is a trial-level constant, and target-split rows duplicate the same
# comparison (identical OS HR and geography), so they are dropped from geography models.
# Arm-split rows are retained: same geography, but genuinely different OS HRs.
# Panel 2A above is unaffected: there the two rows carry different crossover values.
df_fig2_geo = build_dedup_geo_df(df_fig2, TARGET_SPLIT_DROPS)
m2 = df_fig2_geo[df_fig2_geo["Combined_OS_HR"].notna() & df_fig2_geo["Combined_OS_HR_CI_low"].notna() & df_fig2_geo["se_log_hr"].notna() & (df_fig2_geo["se_log_hr"] > 0) & df_fig2_geo[col_geo].notna()].copy()
y2 = m2["log_hr"].values
v2 = m2["v"].values
x2_pct = pd.to_numeric(m2[col_geo], errors="coerce").values
x2_scaled = x2_pct / 10.0
X2 = np.column_stack([np.ones(len(y2)), x2_scaled])
res2 = meta_reg(y2, v2, X2)

w2 = 1.0 / (v2 + res2["tau2"])
weights_norm2 = 20.0 + (w2 - w2.min()) / (w2.max() - w2.min() + 1e-12) * 180.0

ax2.scatter(x2_pct, y2, s=weights_norm2, color="#2ca02c", alpha=0.6, edgecolors="black", linewidth=0.5, zorder=3)

x_grid2_pct = np.linspace(x2_pct.min(), x2_pct.max(), 200)
x_grid2_scaled = x_grid2_pct / 10.0
X_grid2 = np.column_stack([np.ones(len(x_grid2_pct)), x_grid2_scaled])
y_fit2 = X_grid2 @ res2["beta"]

se_fit2 = np.sqrt(np.sum((X_grid2 @ (res2["q"] * res2["XtWX_inv"])) * X_grid2, axis=1))
t_crit2 = stats.t.ppf(0.975, df=res2["k"]-2)
ci_low2 = y_fit2 - t_crit2 * se_fit2
ci_high2 = y_fit2 + t_crit2 * se_fit2

ax2.plot(x_grid2_pct, y_fit2, color="grey", linestyle="--", linewidth=2, zorder=4)
ax2.fill_between(x_grid2_pct, ci_low2, ci_high2, color="#2ca02c", alpha=0.15, zorder=2)

ax2.set_yticks(yticks)
ax2.set_yticklabels(yticklabels)
ax2.set_xlabel("% Participants Enrolled in North America or Europe", fontsize=11, fontweight="bold")
ax2.set_title("B  Western Enrollment vs Overall Survival", fontsize=12, fontweight="bold", loc="left", pad=10)

p2 = res2["p_hk"][1]

stats_text2 = f"Number of comparisons = {res2['k']}\n{fmt_p_jama(p2)}"
ax2.text(0.04, 0.94, stats_text2, transform=ax2.transAxes, fontsize=9.0, va="top", ha="left",
         bbox=dict(boxstyle="round,pad=0.5", facecolor="white", edgecolor="#cccccc", alpha=0.95), zorder=5)

for ax in (ax1, ax2):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="out")

out_dir = Path(FIGURES_DIR)
png_fig2 = out_dir / "Figure_2.png"
pdf_fig2 = out_dir / "Figure_2.pdf"
fig.savefig(png_fig2, dpi=600, bbox_inches="tight")
fig.savefig(pdf_fig2, bbox_inches="tight", metadata={"CreationDate": None})   # vector, for submission; no timestamp
plt.close(fig)

print(f"Saved Figure 2 to {png_fig2} and {pdf_fig2}")

Saved Figure 2 to outputs/figures/Figure_2.png and outputs/figures/Figure_2.pdf
